
# Hybrid Retrieval for Agent Memory: Vector, Lexical, and Metadata Together

Companion notebook for the Oracle AI Database article *Hybrid Retrieval for Agent Memory: Vector, Lexical, and Metadata Together*. It runs on the schema from the earlier articles in this series: From Prompt to Persistence [part 1](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-1-designing-multi-tenant-agent-memory-schemas-for-saas) and [part 2](https://blogs.oracle.com/developers/from-prompt-to-persistence-part-2-putting-the-multi-tenant-agent-memory-schema-to-work) (companion notebook: [multitenant_schema_walkthrough.ipynb](multitenant_schema_walkthrough.ipynb)), and on the canonical/derived split from [Persistent Memory and Derived Context](https://blogs.oracle.com/developers/persistent-memory-and-derived-context-a-two-layer-pattern-for-agents) (companion notebook: [two_layer_pattern_walkthrough.ipynb](two_layer_pattern_walkthrough.ipynb)).

Part 2's converged retrieval query left a vector score and a lexical score side by side.
This notebook combines them, then follows the surviving candidates through reranking and
context budgeting.

The pipeline has six stages. Stages 2 through 5 run in **one SQL statement**. Section 5
builds the filtering, candidate generation, and fusion stages and shows their execution
plan; Section 6 adds the in-database reranker to the same statement.

| Stage | What it does | Where it runs |
| --- | --- | --- |
| 1. Query understanding | entities, expanded vocabulary, intent | application (§4) |
| 2. Metadata filtering | scope, lifecycle, collection, section | SQL (§5) |
| 3. Candidate generation | vector pool + lexical pool | SQL (§5) |
| 4. Score fusion | reciprocal rank fusion | SQL (§5) |
| 5. Reranking | cross-encoder over (query, chunk) pairs | in-database (§6) |
| 6. Context budgeting | diversity, near-duplicates, score cliff | application (§7) |

## Prerequisites

- Oracle AI Database 26ai at `localhost:1521/FREEPDB1` (or set `DB_DSN`). The notebook
  creates an HNSW vector index, which requires an enabled Vector Pool. Autonomous AI
  Database manages this automatically. For a local or other non-Autonomous database with
  `SGA_TARGET > 0`, a DBA can enable automatic sizing with
  `ALTER SYSTEM SET vector_memory_size = 1 SCOPE=SPFILE;`, followed by a database restart.
- A user with `DB_DEVELOPER_ROLE`, `CREATE SESSION`, `CREATE TABLE`, `EXECUTE ON DBMS_RLS`.
- `python-oracledb` 2.5 or newer in Thin mode (the default), which supports the Developer Hub program identifier.
- The ONNX embedding model `ALL_MINILM_L12_V2` (384-dim) loaded.
- A reranking model loaded as `BGE_RERANKER`. See §6 for how to produce and load it.
- A `.env` with `DB_USER`, `DB_PASSWORD`, `DB_DSN` at the repository root, and Jupyter
  started from the `notebooks/` folder so `load_dotenv()` can walk up and find it.
- The **Article 5 (hybrid retrieval)** kernel selected. The default `python3` kernel does
  not include this notebook's dependencies.

Vectors are `VECTOR(384, FLOAT32)` to match `ALL_MINILM_L12_V2`.


## 1. Environment & connection

In [1]:

import hashlib
import json
import os
import re
import time
import uuid
from datetime import datetime, timezone

import oracledb
from dotenv import load_dotenv

# Identify this notebook in Oracle's client metadata before creating a connection.
oracledb.defaults.program = "devrel-developerhub-hybrid-retrieval-for-agent-memory-vector-lexical-and-metadata-together"

# Return CLOB columns (chunk content) as str rather than LOB locators so previews print.
oracledb.defaults.fetch_lobs = False

# load_dotenv() searches upward from the kernel's working directory, so this notebook
# has to be launched from the repo's notebooks/ folder for it to find ../.env.
load_dotenv()

if not os.getenv("DB_USER"):
    raise RuntimeError(
        "No DB_USER in the environment. This notebook reads credentials from the "
        "repository .env, which load_dotenv() finds by walking up from the working "
        "directory, so Jupyter has to be started from the notebooks/ folder: "
        "cd <repo>/notebooks && jupyter lab hybrid_retrieval_pipeline.ipynb "
        "(expected keys: DB_USER, DB_PASSWORD, DB_DSN)."
    )

DB_USER = os.environ["DB_USER"]
DB_PASSWORD = os.environ["DB_PASSWORD"]
DB_DSN = os.getenv("DB_DSN", "localhost:1521/FREEPDB1")

conn = oracledb.connect(user=DB_USER, password=DB_PASSWORD, dsn=DB_DSN)
cur = conn.cursor()

# Pin the session to UTC. valid_from/valid_until/created_at are plain TIMESTAMP written
# from SYSTIMESTAMP (DB tz = UTC); a session in another zone misjudges superseded rows.
cur.execute("ALTER SESSION SET TIME_ZONE = 'UTC'")

TENANT_ACME = "tenant_acme"
TENANT_GLOBEX = "tenant_globex"
USER_JANE = "user_jane"
AGENT_ID = "agent:research_v1"


def new_id(prefix: str) -> str:
    return f"{prefix}_{uuid.uuid4().hex[:12]}"


def sha256(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


print("Oracle version:", conn.version)

Oracle version: 23.26.1.0.0



## 2. The knowledge-base tables, the indexes, and the tenant boundary

This notebook is self-contained: it builds the subset of the part 1 schema it needs
(`knowledge_base_document` and `knowledge_base_chunk`) rather than depending on a previous
notebook's leftovers. The DDL is unchanged from part 1.

Each index supports a different part of the pipeline:

- `idx_kb_doc_collection`: a plain B-tree that Stage 2's metadata filter rides on.
- `idx_kb_chunk_embedding`: an in-memory neighbor graph (HNSW) available to the optimizer for Stage 3's vector pool.
- `idx_kb_chunk_text`: an Oracle Text index for Stage 3's lexical pool. **This is the one
  piece of new DDL the article introduces.** Entity and summarization memory already carried
  Oracle Text indexes from part 1; the knowledge-base chunks did not, because part 1's
  knowledge-base branch was vector-only.

In [2]:

# Idempotent teardown so the notebook can be re-run from the top.
for stmt in [
    "BEGIN DBMS_RLS.DROP_POLICY(USER, 'KNOWLEDGE_BASE_CHUNK', 'KB_CHUNK_TENANT_POL'); END;",
    "BEGIN DBMS_RLS.DROP_POLICY(USER, 'KNOWLEDGE_BASE_DOCUMENT', 'KB_DOC_TENANT_POL'); END;",
    "DROP TABLE knowledge_base_chunk CASCADE CONSTRAINTS PURGE",
    "DROP TABLE knowledge_base_document CASCADE CONSTRAINTS PURGE",
]:
    try:
        cur.execute(stmt)
    except oracledb.DatabaseError:
        pass  # first run

cur.execute("""
CREATE TABLE knowledge_base_document (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  user_id         VARCHAR2(64),
  agent_id        VARCHAR2(64),
  collection      VARCHAR2(128) NOT NULL,
  title           VARCHAR2(512) NOT NULL,
  source_uri      VARCHAR2(1024),
  source_type     VARCHAR2(64)  NOT NULL,
  metadata        JSON,
  ingested_at     TIMESTAMP     NOT NULL,
  ingested_by     VARCHAR2(64)  NOT NULL,
  version         NUMBER(10)    NOT NULL,
  superseded_by   VARCHAR2(64),
  valid_from      TIMESTAMP     NOT NULL,
  valid_until     TIMESTAMP,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP
)
""")

cur.execute("""
CREATE TABLE knowledge_base_chunk (
  id              VARCHAR2(64)  PRIMARY KEY,
  tenant_id       VARCHAR2(64)  NOT NULL,
  document_id     VARCHAR2(64)  NOT NULL,
  chunk_index     NUMBER(10)    NOT NULL,
  content         CLOB          NOT NULL,
  embedding       VECTOR(384, FLOAT32),
  metadata        JSON,
  created_at      TIMESTAMP     NOT NULL,
  deleted_at      TIMESTAMP,
  CONSTRAINT fk_kb_chunk_doc FOREIGN KEY (document_id)
    REFERENCES knowledge_base_document (id)
)
""")
print("Tables created.")

Tables created.


In [3]:

# Stage 2 rides this one: collection + scope + lifecycle, all leading columns.
cur.execute("""
CREATE INDEX idx_kb_doc_collection
  ON knowledge_base_document (tenant_id, user_id, agent_id, collection, valid_until)
""")

cur.execute("""
CREATE UNIQUE INDEX idx_kb_chunk_doc_idx
  ON knowledge_base_chunk (tenant_id, document_id, chunk_index)
""")

# A functional index on the JSON path, so Stage 2's section_type predicate is indexable
# rather than a per-row JSON parse (the part 1 pattern).
cur.execute("""
CREATE INDEX idx_kb_chunk_section
  ON knowledge_base_chunk (JSON_VALUE(metadata, '$.section_type'))
""")

# Stage 3a: the vector pool.
cur.execute("""
CREATE VECTOR INDEX idx_kb_chunk_embedding
  ON knowledge_base_chunk (embedding)
  ORGANIZATION INMEMORY NEIGHBOR GRAPH
  DISTANCE COSINE
  WITH TARGET ACCURACY 95
""")

# Stage 3b: the lexical pool. The one piece of new DDL in the article. Same form as
# idx_entity_text and idx_summ_text from part 1.
cur.execute("""
CREATE INDEX idx_kb_chunk_text ON knowledge_base_chunk (content)
  INDEXTYPE IS CTXSYS.CONTEXT PARAMETERS ('SYNC (ON COMMIT)')
""")

print("Indexes created: B-tree, JSON functional, HNSW vector, Oracle Text.")

Indexes created: B-tree, JSON functional, HNSW vector, Oracle Text.



### 2a. Row-level security on both knowledge-base tables

The article intentionally leaves `tenant_id` out of the query. RLS appends the predicate
to every reference of both tables, in both candidate
pools and on the final join, so there is no place in a multi-table retrieval where the
filter can be forgotten.

Note both tables get a policy. Part 1's original RLS loop matched `%_MEMORY`, which
silently skipped the knowledge-base tables and left them unisolated.

In [4]:

# Context namespace bound to a package, which is how Oracle scopes who may set it.
cur.execute("""
CREATE OR REPLACE PACKAGE set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant VARCHAR2);
END set_memory_ctx;
""")
cur.execute("""
CREATE OR REPLACE PACKAGE BODY set_memory_ctx AS
  PROCEDURE set_tenant(p_tenant VARCHAR2) IS
  BEGIN
    DBMS_SESSION.SET_CONTEXT('memory_ctx', 'tenant_id', p_tenant);
  END;
END set_memory_ctx;
""")
cur.execute("CREATE OR REPLACE CONTEXT memory_ctx USING set_memory_ctx")

cur.execute("""
CREATE OR REPLACE FUNCTION memory_tenant_policy(
  schema_name IN VARCHAR2, table_name IN VARCHAR2
) RETURN VARCHAR2 AS
BEGIN
  RETURN 'tenant_id = SYS_CONTEXT(''memory_ctx'', ''tenant_id'')';
END;
""")

# update_check => TRUE is load-bearing: without it Oracle rejects INSERT in
# statement_types (ORA-28104), and a row with the wrong tenant_id would not be blocked.
for tbl, pol in [("KNOWLEDGE_BASE_DOCUMENT", "KB_DOC_TENANT_POL"),
                 ("KNOWLEDGE_BASE_CHUNK", "KB_CHUNK_TENANT_POL")]:
    cur.callproc("DBMS_RLS.ADD_POLICY", [], {
        "object_schema": DB_USER.upper(),
        "object_name": tbl,
        "policy_name": pol,
        "policy_function": "memory_tenant_policy",
        "statement_types": "SELECT,INSERT,UPDATE,DELETE",
        "update_check": True,
    })

cur.callproc("set_memory_ctx.set_tenant", [TENANT_ACME])
cur.execute("SELECT SYS_CONTEXT('memory_ctx','tenant_id') FROM dual")
print("RLS active on both KB tables. Session tenant:", cur.fetchone()[0])

RLS active on both KB tables. Session tenant: tenant_acme



## 3. The corpus

Jane's question is the one the article uses throughout:

> "What did the Letta paper say about memory eviction, and how does that compare to what
> the AgentCore docs recommend?"

The corpus is designed to answer that question while exposing the failure modes discussed
in the article:

- The two documents Jane names, in two different collections (`ingested-papers`, `vendor-docs`).
- Near-duplicate eviction chunks inside the Letta paper. Without redundancy at the top of
  the candidate list, reranking has nothing to fix, and the article's claim that reranking
  moves NDCG while leaving recall flat cannot be demonstrated.
- A references chunk that repeats the token "Letta" while answering nothing. This is what
  the `section_type <> 'references'` filter is for; a corpus without one lets that predicate
  pass while proving nothing.
- Distractor documents about agent memory in general, which is what pure vector search returns
  instead of the two named sources.

The chunk text below is **illustrative paraphrase written for this notebook**. Nothing is quoted from the real sources. `source_uri` points at the genuine document, which is what the part 1
schema intends: the database holds a retrievable derived form plus a pointer home, and the
original bytes stay in object storage.

In [5]:
# (collection, doc_key, title, source_uri, source_type)
DOCUMENTS = [
    # -- the two documents Jane names ----------------------------------------
    ("ingested-papers", "letta", "Letta: Agent Memory Beyond the Context Window",
     "https://arxiv.org/abs/2310.08560", "pdf"),
    ("vendor-docs", "agentcore", "Amazon Bedrock AgentCore Memory Developer Guide",
     "https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html", "html"),

    # -- hard negatives: same subject matter, different source ---------------
    ("ingested-papers", "memgpt", "MemGPT: Towards LLMs as Operating Systems",
     "https://arxiv.org/abs/2310.08560", "pdf"),
    ("ingested-papers", "ctxmgmt", "Context Window Management in Long-Running Agents",
     "https://example.org/papers/context-window-management", "pdf"),
    ("ingested-papers", "summarization", "Conversation Summarization Techniques for Dialogue Agents",
     "https://example.org/papers/conversation-summarization", "pdf"),
    ("ingested-papers", "longterm", "Scalable Long-Term Memory for Conversational Agents",
     "https://example.org/papers/scalable-long-term-memory", "pdf"),
    ("ingested-papers", "longctx", "Sliding Window Attention and Long-Context Transformers",
     "https://example.org/papers/long-context-transformers", "pdf"),
    ("ingested-papers", "episodic", "Episodic Memory for Embodied and Situated Agents",
     "https://example.org/papers/episodic-memory-agents", "pdf"),
    ("ingested-papers", "ragdialog", "Retrieval-Augmented Dialogue Systems",
     "https://example.org/papers/retrieval-augmented-dialogue", "pdf"),
    ("ingested-papers", "forgetting", "Forgetting Mechanisms in Neural Memory Systems",
     "https://example.org/papers/forgetting-mechanisms", "pdf"),

    # -- retrieval / evaluation literature -----------------------------------
    ("ingested-papers", "ragsurvey", "A Survey of Retrieval-Augmented Generation for Large Language Models",
     "https://arxiv.org/abs/2312.10997", "pdf"),
    ("ingested-papers", "dpr", "Dense Passage Retrieval for Open-Domain Question Answering",
     "https://arxiv.org/abs/2004.04906", "pdf"),
    ("ingested-papers", "bm25", "BM25 and Lexical Retrieval Revisited",
     "https://example.org/papers/bm25-revisited", "pdf"),
    ("ingested-papers", "reranking", "Cross-Encoder Reranking for Passage Ranking",
     "https://example.org/papers/cross-encoder-reranking", "pdf"),
    ("ingested-papers", "evalmetrics", "Evaluating Retrieval: NDCG, MRR, and Recall",
     "https://example.org/papers/evaluating-retrieval", "pdf"),
    ("ingested-papers", "chunking", "Chunking Strategies for Retrieval-Augmented Generation",
     "https://example.org/papers/chunking-strategies", "pdf"),
    ("ingested-papers", "hyde", "Query Expansion and Hypothetical Document Embeddings",
     "https://example.org/papers/query-expansion-hyde", "pdf"),

    # -- vendor documentation -------------------------------------------------
    ("vendor-docs", "langgraph", "LangGraph Checkpointing and Persistence Guide",
     "https://langchain-ai.github.io/langgraph/concepts/persistence/", "html"),
    ("vendor-docs", "oraclevs", "Oracle AI Vector Search Developer Guide",
     "https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/", "html"),
    ("vendor-docs", "vectortuning", "Vector Index Tuning Guide",
     "https://example.org/docs/vector-index-tuning", "html"),
    ("vendor-docs", "sessionstate", "In-Memory Session State for LLM Applications",
     "https://example.org/docs/session-state", "html"),
    ("vendor-docs", "metadatafilter", "Namespace and Metadata Filtering Reference",
     "https://example.org/docs/metadata-filtering", "html"),
    ("vendor-docs", "multitenant", "Multi-Tenant Data Isolation Patterns",
     "https://example.org/docs/multi-tenant-isolation", "html"),
]

# (doc_key, section_type, text)
CHUNKS = [
    # ======================= LETTA (the paper Jane names) ====================
    ("letta", "abstract",
     "Letta is an agent framework that treats the context window as a managed resource "
     "rather than a fixed budget. It descends from the MemGPT line of work, which framed "
     "the language model as an operating system kernel paging information between a small "
     "fast tier and a large slow tier."),
    ("letta", "introduction",
     "An agent that runs for weeks accumulates far more conversation than any context "
     "window can hold. Naive truncation drops the oldest turns and with them the reasons "
     "behind earlier decisions, so the agent contradicts itself over long horizons."),
    ("letta", "introduction",
     "The contribution is a memory hierarchy the model administers itself. Rather than an "
     "external process deciding what the agent remembers, the agent issues function calls "
     "to read and write its own memory tiers."),

    # --- the chunk that actually answers Jane's question --------------------
    ("letta", "memory-management",
     "Letta frames eviction as recursive summarization. When the working context approaches "
     "its limit, the oldest span of messages is compressed into a summary that is written "
     "back into archival memory, and the raw messages are evicted from the window. The "
     "summary is itself eligible for later compression, so the process is recursive: "
     "summaries of summaries preserve a decaying trace of the entire history."),

    # --- near-duplicates: what crowds the top of a vector-only candidate list
    ("letta", "memory-management",
     "Eviction in Letta is driven by a pressure signal on the working context. As the "
     "context fills, the manager selects the oldest message span, produces a compressed "
     "summary of it, and removes the original messages from the window. Because summaries "
     "can themselves be summarized, the scheme degrades gracefully over long sessions."),
    ("letta", "memory-management",
     "The memory manager evicts messages under context pressure. A span of older turns is "
     "replaced by a generated summary, which is persisted to archival storage while the raw "
     "turns leave the working set. Repeated application yields recursive summarization, the "
     "mechanism Letta relies on for unbounded conversations."),
    ("letta", "memory-management",
     "Context pruning proceeds by recursive summarization: the oldest messages are distilled "
     "into a compact summary, the originals are evicted, and the summary re-enters the window "
     "in their place. Subsequent rounds may compress that summary again."),
    ("letta", "memory-management",
     "When the token count crosses a configured threshold, Letta triggers a flush. The flush "
     "summarizes and evicts the oldest portion of the message buffer, freeing room while "
     "retaining a lossy record of what was removed."),

    ("letta", "architecture",
     "Memory in Letta is tiered. Main context holds the system prompt, a working set of "
     "recent messages, and a small editable scratchpad. External context holds archival "
     "storage and a recall database, both reachable through function calls."),
    ("letta", "architecture",
     "Memory blocks are named, size-bounded regions of the prompt the agent can rewrite. A "
     "persona block and a human block are conventional, and additional blocks can be defined "
     "per deployment."),
    ("letta", "architecture",
     "Archival memory is an append-mostly vector store the agent searches by similarity. "
     "Recall memory indexes the full message history so specific past turns can be retrieved "
     "verbatim rather than through a summary."),
    ("letta", "evaluation",
     "The paper evaluates on multi-session dialogue and document question answering, "
     "reporting that tiered memory with recursive summarization sustains coherence across "
     "sessions far longer than a fixed truncation baseline."),
    ("letta", "evaluation",
     "Ablations remove the recursive step and summarize only once. Coherence degrades sharply "
     "on the longest sessions, which the authors attribute to unbounded growth of the single "
     "summary rather than to information loss per se."),
    ("letta", "discussion",
     "A limitation is that summarization is lossy in ways the agent cannot detect. Once a "
     "detail is compressed away it is unrecoverable from the window, and the agent has no "
     "signal that it is reasoning over a lossy trace."),

    # --- the references chunk: matches "Letta" repeatedly, answers nothing ---
    ("letta", "references",
     "References. Packer et al. Letta and MemGPT: Towards LLMs as Operating Systems. See also "
     "the Letta documentation, the Letta framework repository, the Letta agent development "
     "guide, the Letta memory blocks reference, Letta deployment notes, and subsequent Letta "
     "papers on agent state. Letta, Letta, Letta."),

    # ======================= AGENTCORE (the vendor docs) =====================
    ("agentcore", "overview",
     "Amazon Bedrock AgentCore Memory gives agents both short-term and long-term memory. "
     "Short-term memory holds the raw events of a session; long-term memory holds durable "
     "records extracted from those events by configurable strategies."),
    ("agentcore", "overview",
     "A memory resource is the top-level container. Creating one establishes the namespaces, "
     "the extraction strategies, and the retention configuration that govern everything "
     "written beneath it."),

    # --- the chunk that answers the AgentCore half of the question ----------
    # Deliberately written in AWS's own vocabulary: expiry, retention, extraction
    # strategies. It never says "eviction", "pruning" or "summarizing under pressure",
    # because the real documentation doesn't. This is the vocabulary gap the article is
    # about -- a vendor describing the same lifecycle concern in entirely different words.
    # Phrasing this chunk in the questioner's terms would make it trivially findable and
    # would quietly turn the whole evaluation into a tautology.
    ("agentcore", "retention",
     "Each memory resource defines an event expiry period, specified in days. When that "
     "period lapses the raw session events are removed automatically. Records that the "
     "extraction strategies have already promoted into long-term memory are unaffected, so "
     "configuring strategies is how you determine what outlives a session."),

    ("agentcore", "retention",
     "Retention is configured per memory resource rather than per session. The setting "
     "governs the short-term event stream only; long-term records persist independently of "
     "the events they were derived from."),
    ("agentcore", "strategies",
     "Built-in extraction strategies cover semantic facts, user preferences, and session "
     "summaries. A custom strategy can override the prompt used for extraction."),
    ("agentcore", "strategies",
     "Strategies run asynchronously after events are written, so long-term memory is "
     "eventually consistent with the session. An application that reads immediately after "
     "writing may not see extracted records yet."),
    ("agentcore", "namespaces",
     "Long-term memories are organized by namespace, which scopes a record to an actor, a "
     "session, or a broader grouping. Retrieval takes a namespace and a query string and "
     "returns semantically matching records."),
    ("agentcore", "namespaces",
     "Namespace templates interpolate identifiers such as actor and session, so a single "
     "strategy definition can partition memories per user without enumerating them."),
    ("agentcore", "short-term",
     "Short-term memory is retrieved as the recent turns of a session, in order, and is "
     "intended to be passed to the model directly as conversational context."),
    ("agentcore", "short-term",
     "Events are appended with an actor identifier and a role. The service does not "
     "summarize or compress the event stream; it stores what the application writes until "
     "expiry."),
    ("agentcore", "integration",
     "The memory APIs are independent of the agent runtime, so an agent hosted elsewhere can "
     "still use AgentCore Memory by calling the create-event and retrieve-memory operations "
     "directly."),

    # ======================= MEMGPT (closest hard negative) ==================
    ("memgpt", "abstract",
     "MemGPT introduces a virtual context management scheme inspired by operating system "
     "paging. The model moves information between an in-context working set and out-of-context "
     "storage through explicit function calls it decides to issue."),
    ("memgpt", "introduction",
     "Fixed context windows are the binding constraint on conversational agents. Rather than "
     "enlarging the window, MemGPT treats it as physical memory and adds a paging layer."),
    ("memgpt", "memory-management",
     "When the context window fills, MemGPT flushes older messages to external storage after "
     "generating a recursive summary, keeping a pointer so the detail can be paged back in "
     "on demand."),
    ("memgpt", "memory-management",
     "A memory pressure warning is injected into the context when utilization crosses a "
     "threshold, prompting the model to take an eviction action before it is forced."),
    ("memgpt", "architecture",
     "Main context is divided into system instructions, working context, and the FIFO message "
     "queue. External context comprises recall storage and archival storage."),
    ("memgpt", "evaluation",
     "Evaluation covers deep memory retrieval and nested key-value lookup, showing that paged "
     "virtual context outperforms a fixed window on tasks requiring old information."),
    ("memgpt", "discussion",
     "The authors note that function-calling reliability bounds the approach: an agent that "
     "fails to issue a paging call at the right moment loses the information regardless of "
     "what storage holds."),

    # ======================= CONTEXT WINDOW MANAGEMENT =======================
    ("ctxmgmt", "abstract",
     "Long-running agents must decide continuously what to keep in the context window. This "
     "survey compares truncation, summarization, retrieval, and hybrid schemes across several "
     "open agent frameworks."),
    ("ctxmgmt", "methods",
     "Sliding-window truncation is the cheapest policy and the most lossy. Summarization "
     "trades compute for retention. Retrieval-based schemes keep the window small and pull "
     "detail back on demand, at the cost of a retrieval failure mode."),
    ("ctxmgmt", "methods",
     "Hybrid policies summarize the middle of a session while preserving the first and last "
     "turns verbatim, on the observation that openings carry task framing and recent turns "
     "carry active state."),
    ("ctxmgmt", "methods",
     "Eviction policies borrowed from cache design — least recently used, least frequently "
     "referenced — have been applied to message spans, with mixed results because relevance "
     "in dialogue is not well predicted by recency alone."),
    ("ctxmgmt", "evaluation",
     "The authors measure coherence over synthetic 200-turn sessions, scoring whether the "
     "agent contradicts commitments it made earlier."),
    ("ctxmgmt", "discussion",
     "No single policy dominates. Agents with bounded sessions do well with truncation; agents "
     "with open-ended relationships to a user need durable memory outside the window."),
    ("ctxmgmt", "discussion",
     "The survey argues that context management and retrieval are the same problem viewed "
     "from two directions, and that treating them separately produces systems that summarize "
     "away exactly what retrieval would later need."),

    # ======================= SUMMARIZATION ===================================
    ("summarization", "abstract",
     "Dialogue summarization condenses conversational history while preserving entities, "
     "commitments, and unresolved threads. Abstractive methods generate new text; extractive "
     "methods select salient turns."),
    ("summarization", "methods",
     "Hierarchical summarization compresses in stages, summarizing summaries as a session "
     "grows. Fidelity degrades with each stage, so entity anchoring is commonly used to "
     "preserve names and numbers through successive rounds."),
    ("summarization", "methods",
     "Incremental summarization updates a running summary after each turn rather than "
     "recomputing from the full history, trading fidelity for bounded cost per turn."),
    ("summarization", "evaluation",
     "Reference-based metrics correlate poorly with downstream task success, so the paper "
     "advocates measuring whether the summary preserves the information a later question "
     "actually needs."),
    ("summarization", "discussion",
     "Compression is not lossless and the loss is not uniform: numbers, negations, and "
     "conditionals are dropped disproportionately relative to their importance."),

    # ======================= LONG-TERM MEMORY ================================
    ("longterm", "abstract",
     "This work proposes a memory layer that extracts durable facts from conversation and "
     "stores them independently of the session, so an agent can recall a user's preferences "
     "across unrelated sessions."),
    ("longterm", "methods",
     "Extraction runs asynchronously over completed turns, producing candidate facts that are "
     "deduplicated against existing memory before being written."),
    ("longterm", "methods",
     "Conflicting facts are resolved by recency with a confidence weight, and superseded "
     "facts are retained with a validity interval rather than deleted."),
    ("longterm", "methods",
     "Retrieval is a similarity search over the fact store, filtered by the user the facts "
     "belong to. The paper stresses that the filter is a correctness requirement, not an "
     "optimization."),
    ("longterm", "evaluation",
     "On a multi-session benchmark the memory layer improves answer accuracy on questions "
     "whose evidence appeared in an earlier session."),
    ("longterm", "discussion",
     "The authors observe that extraction quality, not storage or retrieval, is the binding "
     "constraint: facts never extracted cannot be recalled no matter how good the index is."),

    # ======================= LONG CONTEXT ====================================
    ("longctx", "abstract",
     "Sliding window attention bounds the quadratic cost of self-attention by restricting "
     "each token's receptive field, enabling far longer sequences at a fixed compute budget."),
    ("longctx", "methods",
     "Dilated and global-token variants restore some long-range connectivity by letting a "
     "small number of positions attend everywhere."),
    ("longctx", "evaluation",
     "Retrieval accuracy within very long contexts is uneven: models attend strongly to the "
     "beginning and end of the window and weakly to the middle."),
    ("longctx", "discussion",
     "A larger window does not remove the need to choose what goes in it. Degraded attention "
     "in the middle of long contexts means ordering still determines what the model uses."),
    ("longctx", "discussion",
     "The paper cautions against reading long-context benchmarks as evidence that retrieval "
     "is unnecessary, since benchmark passages are typically placed adversarially rather than "
     "assembled from a noisy corpus."),

    # ======================= EPISODIC MEMORY =================================
    ("episodic", "abstract",
     "Episodic memory stores specific past experiences with their temporal and situational "
     "context, in contrast to semantic memory which stores decontextualized facts."),
    ("episodic", "methods",
     "Episodes are segmented at boundaries detected from changes in task or location, then "
     "indexed by embedding along with their timestamps."),
    ("episodic", "methods",
     "Consolidation periodically promotes recurring patterns from episodic traces into "
     "semantic memory, a process loosely modeled on sleep-dependent consolidation."),
    ("episodic", "evaluation",
     "Agents with episodic recall outperform flat-memory baselines on tasks requiring the "
     "agent to remember how a similar situation was handled previously."),
    ("episodic", "discussion",
     "Unbounded episodic growth forces a forgetting policy, and the authors note that the "
     "choice of what to forget is effectively a choice about what the agent can become."),

    # ======================= RETRIEVAL-AUGMENTED DIALOGUE ====================
    ("ragdialog", "abstract",
     "Retrieval-augmented dialogue grounds each response in passages fetched at turn time, "
     "reducing hallucination relative to closed-book generation."),
    ("ragdialog", "methods",
     "Query construction is the hard part in dialogue: the user's latest turn is often "
     "elliptical, so the retrieval query must be rewritten using conversational context."),
    ("ragdialog", "methods",
     "Rewriting can be learned or prompted. Prompted rewriting with a small fast model is "
     "the common production choice because it adds a bounded, predictable latency."),
    ("ragdialog", "evaluation",
     "Groundedness improves with retrieval quality but saturates: beyond a point, adding "
     "passages dilutes the prompt and accuracy falls."),
    ("ragdialog", "discussion",
     "The authors argue that dialogue systems should retrieve from their own conversation "
     "history as well as from a document corpus, treating both as one ranked pool."),

    # ======================= FORGETTING ======================================
    ("forgetting", "abstract",
     "Forgetting is treated here as a design parameter rather than a failure. A memory system "
     "that never forgets accumulates stale and contradictory content that degrades retrieval."),
    ("forgetting", "methods",
     "Decay functions age memories by time since last access, so frequently used memories "
     "persist and unused ones fall below a retention threshold."),
    ("forgetting", "methods",
     "Explicit invalidation marks a memory superseded when a contradicting fact arrives, "
     "preserving the old value with a closed validity interval for audit."),
    ("forgetting", "evaluation",
     "Aggressive forgetting improves precision and harms recall; the paper reports the "
     "trade-off curve across several decay rates."),
    ("forgetting", "discussion",
     "The authors distinguish eviction from a working buffer, which is about capacity, from "
     "forgetting in durable storage, which is about correctness. Conflating the two produces "
     "systems that delete durable facts for capacity reasons."),

    # ======================= RAG SURVEY ======================================
    ("ragsurvey", "abstract",
     "Retrieval-augmented generation grounds model output in retrieved passages. This survey "
     "organizes the literature into naive, advanced, and modular RAG paradigms."),
    ("ragsurvey", "retrieval",
     "Advanced RAG adds pre-retrieval query rewriting and post-retrieval reranking. Reranking "
     "with a cross-encoder consistently improves ordering over bi-encoder similarity alone."),
    ("ragsurvey", "retrieval",
     "Hybrid retrieval combining sparse and dense signals outperforms either alone on "
     "heterogeneous corpora, particularly where rare identifiers matter."),
    ("ragsurvey", "retrieval",
     "Fusion methods range from score normalization to rank-based combination. Rank-based "
     "fusion avoids the calibration problem entirely and is difficult to beat without "
     "corpus-specific tuning."),
    ("ragsurvey", "evaluation",
     "Common metrics are recall at k for the retriever and faithfulness or answer relevance "
     "for the generator. The survey notes that retrieval quality bounds end-to-end quality."),
    ("ragsurvey", "discussion",
     "Modular RAG decomposes the pipeline into interchangeable stages, which the survey "
     "argues is a prerequisite for measuring any single stage's contribution."),

    # ======================= DPR =============================================
    ("dpr", "abstract",
     "Dense passage retrieval learns a bi-encoder that maps questions and passages into a "
     "shared space, retrieving by inner product rather than lexical overlap."),
    ("dpr", "methods",
     "Training uses in-batch negatives plus mined hard negatives, which the paper finds "
     "essential: without hard negatives the encoder learns only coarse topical similarity."),
    ("dpr", "methods",
     "Question and passage encoders are separate towers, so passage embeddings can be "
     "precomputed and indexed offline. This is what makes the approach scalable."),
    ("dpr", "evaluation",
     "Dense retrieval beats BM25 on natural-language questions but underperforms on queries "
     "dominated by rare entities or exact strings."),
    ("dpr", "discussion",
     "The independence of the two towers is both the source of scalability and the ceiling "
     "on precision, since neither text is encoded with knowledge of the other."),

    # ======================= BM25 ============================================
    ("bm25", "abstract",
     "BM25 remains a strong baseline for retrieval, particularly where queries contain exact "
     "identifiers, code fragments, or error strings."),
    ("bm25", "methods",
     "The scoring function weights term frequency with saturation and penalizes long "
     "documents, with two parameters controlling each effect."),
    ("bm25", "methods",
     "Lexical retrieval fails when the query and document use different vocabulary for the "
     "same concept, a mismatch no amount of parameter tuning repairs."),
    ("bm25", "evaluation",
     "On benchmarks with heavy entity overlap between query and document, BM25 matches or "
     "exceeds dense retrievers trained on far more data."),
    ("bm25", "discussion",
     "The paper argues that lexical and dense retrieval fail on disjoint query populations, "
     "which is the case for combining them rather than choosing between them."),

    # ======================= RERANKING =======================================
    ("reranking", "abstract",
     "Cross-encoder reranking scores a query and a candidate jointly in a single forward "
     "pass, producing a relevance estimate computed with full knowledge of both texts."),
    ("reranking", "methods",
     "Because scoring is per pair, cross-encoders cannot precompute document representations "
     "and cannot be used to search a corpus. They operate only on a candidate set produced "
     "by something cheaper."),
    ("reranking", "methods",
     "Candidate set size controls the quality-latency trade-off. A larger set gives the "
     "reranker more chances to recover a good passage the retriever ranked poorly, at linear "
     "cost in inference time."),
    ("reranking", "evaluation",
     "Reranking improves ordering metrics substantially while leaving recall unchanged, since "
     "it only reorders what it was given."),
    ("reranking", "discussion",
     "The authors note that reranking cannot repair a candidate set that omits the answer, "
     "making retriever recall the binding constraint on the pipeline."),

    # ======================= EVAL METRICS ====================================
    ("evalmetrics", "abstract",
     "This tutorial reviews ranking metrics and the conditions under which each is "
     "informative for retrieval evaluation."),
    ("evalmetrics", "methods",
     "Recall at k measures how many relevant items appear in the top k without regard to "
     "their order, making it the natural metric for a candidate-generation stage."),
    ("evalmetrics", "methods",
     "Normalized discounted cumulative gain rewards placing highly relevant items early, "
     "supports graded relevance, and is the appropriate choice when a downstream consumer "
     "reads results in order."),
    ("evalmetrics", "methods",
     "Mean reciprocal rank considers only the first relevant result and is appropriate when "
     "a single correct answer suffices."),
    ("evalmetrics", "evaluation",
     "Small evaluation sets produce wide confidence intervals; the tutorial recommends "
     "reporting variance and treating differences below it as noise."),
    ("evalmetrics", "discussion",
     "Metric choice should follow how the results are consumed. Optimizing recall when the "
     "consumer reads only the top few is a common and expensive mistake."),

    # ======================= CHUNKING ========================================
    ("chunking", "abstract",
     "Chunk boundaries determine what can be retrieved. This study compares fixed-size, "
     "sentence-aware, and structure-aware chunking across several corpora."),
    ("chunking", "methods",
     "Overlap between adjacent chunks reduces the chance that an answer straddles a boundary, "
     "at the cost of index size and near-duplicate results."),
    ("chunking", "methods",
     "Structure-aware chunking that respects section boundaries preserves the coherence of "
     "each chunk and enables section-level metadata filtering."),
    ("chunking", "evaluation",
     "Retrieval quality is more sensitive to chunk size than to embedding model choice within "
     "the range tested."),
    ("chunking", "discussion",
     "Near-duplicate chunks created by overlap inflate apparent redundancy in results and "
     "should be collapsed after ranking rather than avoided during indexing."),

    # ======================= QUERY EXPANSION =================================
    ("hyde", "abstract",
     "Query expansion rewrites a short query into a richer form before retrieval, closing "
     "part of the vocabulary gap between questions and documents."),
    ("hyde", "methods",
     "Hypothetical document embedding generates a plausible answer and embeds that instead of "
     "the question, on the theory that answers look more like documents than questions do."),
    ("hyde", "methods",
     "Synonym expansion for lexical retrieval adds alternate surface forms for key concepts, "
     "which matters when the corpus uses different terminology than the user."),
    ("hyde", "evaluation",
     "Expansion helps most on short, underspecified queries and can hurt on queries that are "
     "already precise, by diluting the discriminative terms."),
    ("hyde", "discussion",
     "Extracting named entities before expansion and preserving them verbatim protects the "
     "exact-match signal that expansion would otherwise wash out."),

    # ======================= LANGGRAPH =======================================
    ("langgraph", "overview",
     "Checkpointing persists graph state after each node, so a run can be resumed, inspected, "
     "or replayed from any prior step."),
    ("langgraph", "methods",
     "A checkpointer writes state to a backing store keyed by thread. Threads isolate "
     "concurrent conversations from one another."),
    ("langgraph", "methods",
     "State is a typed dictionary whose keys are updated by nodes; reducers control how "
     "concurrent updates to the same key are merged."),
    ("langgraph", "methods",
     "Interrupting before or after a node enables human-in-the-loop review, with the run "
     "resuming from the persisted checkpoint once approved."),
    ("langgraph", "discussion",
     "Persistence at the orchestration layer is distinct from the agent's memory: one records "
     "how the run progressed, the other records what the agent knows."),

    # ======================= ORACLE VECTOR SEARCH ============================
    ("oraclevs", "overview",
     "AI Vector Search adds a native VECTOR data type and distance operators, so similarity "
     "search runs inside SQL alongside relational predicates and joins."),
    ("oraclevs", "methods",
     "VECTOR_EMBEDDING generates an embedding inside the database from a loaded ONNX model, "
     "so query text never leaves the engine to be vectorized."),
    ("oraclevs", "methods",
     "In-memory neighbor graph indexes provide approximate search with a configurable target "
     "accuracy, trading recall against query time."),
    ("oraclevs", "methods",
     "Because vector predicates compose with ordinary SQL predicates, the optimizer chooses "
     "whether to filter before or after the index probe based on statistics."),
    ("oraclevs", "methods",
     "Hybrid search combining vector distance with an Oracle Text CONTAINS score can be "
     "expressed in a single statement, with both pools reading one transactional snapshot."),
    ("oraclevs", "discussion",
     "Keeping retrieval in the database means security policies, snapshots, and query planning "
     "apply to it automatically rather than being reimplemented in application code."),

    # ======================= VECTOR TUNING ===================================
    ("vectortuning", "overview",
     "Vector index tuning balances recall against latency and memory. Graph indexes expose a "
     "target accuracy parameter that governs how much of the graph is explored per query."),
    ("vectortuning", "parameters",
     "Raising target accuracy increases the number of neighbors visited, improving recall at "
     "the cost of query time."),
    ("vectortuning", "parameters",
     "Memory sizing determines whether the graph stays resident. An undersized pool forces "
     "index build failures or falls back to exact search."),
    ("vectortuning", "parameters",
     "Rebuilding is required after a change of embedding model, since the vector space itself "
     "changes and prior neighbors are no longer meaningful."),
    ("vectortuning", "discussion",
     "Accuracy targets should be validated against a labeled set rather than accepted from "
     "defaults, because achievable recall depends on the distribution of the data."),

    # ======================= SESSION STATE ===================================
    ("sessionstate", "overview",
     "In-memory stores are commonly used to hold the active turn buffer for a conversation, "
     "where low latency matters more than durability."),
    ("sessionstate", "methods",
     "Time-to-live on session keys bounds memory growth, with the practical consequence that "
     "anything not promoted to durable storage before expiry is lost."),
    ("sessionstate", "methods",
     "Separating the volatile turn buffer from durable memory avoids the mistake of treating "
     "a cache eviction as a decision about what the agent should remember."),
    ("sessionstate", "discussion",
     "The guide recommends writing durable facts synchronously with the turn that produced "
     "them rather than relying on a background job that may not run before expiry."),

    # ======================= METADATA FILTERING ==============================
    ("metadatafilter", "overview",
     "Metadata filters restrict a similarity search to records matching structured criteria, "
     "applied alongside the vector comparison."),
    ("metadatafilter", "methods",
     "Namespaces partition an index so queries never see records outside the requested "
     "partition, which is the usual mechanism for per-tenant separation."),
    ("metadatafilter", "methods",
     "Filter selectivity affects performance: a highly selective filter applied after an "
     "approximate search can leave too few results, so pre-filtering is preferred where "
     "supported."),
    ("metadatafilter", "discussion",
     "A filter expressed in application code is a filter that can be omitted. Enforcing "
     "partitioning in the store removes that class of mistake."),

    # ======================= MULTI-TENANCY ===================================
    ("multitenant", "overview",
     "Multi-tenant systems must guarantee that one tenant's queries can never return another "
     "tenant's rows, regardless of application behavior."),
    ("multitenant", "methods",
     "Shared-table designs with a tenant column are efficient but rely on every query "
     "carrying the predicate, unless the database enforces it."),
    ("multitenant", "methods",
     "Row-level security attaches the predicate at the engine, so it applies to every "
     "reference of the table including joins and subqueries."),
    ("multitenant", "methods",
     "Session context supplies the tenant identity, and the policy function reads it, which "
     "keeps the boundary out of application SQL entirely."),
    ("multitenant", "discussion",
     "Isolation enforced structurally is auditable in a way that isolation enforced by "
     "convention is not: there is one place to inspect rather than every query in the "
     "codebase."),
]

In [6]:

now = datetime.now(timezone.utc).replace(tzinfo=None)
doc_ids = {}

for collection, key, title, uri, stype in DOCUMENTS:
    did = new_id("kbd")
    doc_ids[key] = did
    cur.execute("""
    INSERT INTO knowledge_base_document (
      id, tenant_id, agent_id, collection, title, source_uri, source_type,
      metadata, ingested_at, ingested_by, version, valid_from, created_at
    ) VALUES (
      :id, :tenant_id, :agent_id, :collection, :title, :uri, :stype,
      JSON(:meta), :now, 'ingest:pipeline', 1, :now, :now
    )
    """, id=did, tenant_id=TENANT_ACME, agent_id=AGENT_ID, collection=collection,
         title=title, uri=uri, stype=stype,
         meta=json.dumps({"doc_key": key}), now=now)

# Chunks carry an inline embedding generated in-database, and a section_type in metadata
# that Stage 2 filters on. chunk_index is assigned per document in corpus order.
per_doc_ix = {}
chunk_ids = []
for key, section, text in CHUNKS:
    ix = per_doc_ix.get(key, 0)
    per_doc_ix[key] = ix + 1
    cid = new_id("kbc")
    chunk_ids.append(cid)
    cur.execute("""
    INSERT INTO knowledge_base_chunk (
      id, tenant_id, document_id, chunk_index, content, embedding, metadata, created_at
    ) VALUES (
      :id, :tenant_id, :doc_id, :ix, :content,
      VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :content AS DATA),
      JSON(:meta), :now
    )
    """, id=cid, tenant_id=TENANT_ACME, doc_id=doc_ids[key], ix=ix,
         content=text, meta=json.dumps({"section_type": section}), now=now)

conn.commit()  # commit triggers the Oracle Text index SYNC (ON COMMIT)

cur.execute("""
SELECT d.collection, d.title, COUNT(c.id)
  FROM knowledge_base_document d
  JOIN knowledge_base_chunk c ON c.document_id = d.id
 GROUP BY d.collection, d.title
 ORDER BY d.collection, d.title
""")
print(f"{'collection':<17} {'chunks':>6}  title")
for coll, title, n in cur:
    print(f"{coll:<17} {n:>6}  {title}")

cur.execute("SELECT COUNT(*) FROM knowledge_base_chunk")
print("\ntotal chunks:", cur.fetchone()[0])

collection        chunks  title
ingested-papers        6  A Survey of Retrieval-Augmented Generation for Large Language Models
ingested-papers        5  BM25 and Lexical Retrieval Revisited
ingested-papers        5  Chunking Strategies for Retrieval-Augmented Generation
ingested-papers        7  Context Window Management in Long-Running Agents
ingested-papers        5  Conversation Summarization Techniques for Dialogue Agents
ingested-papers        5  Cross-Encoder Reranking for Passage Ranking
ingested-papers        5  Dense Passage Retrieval for Open-Domain Question Answering
ingested-papers        5  Episodic Memory for Embodied and Situated Agents
ingested-papers        6  Evaluating Retrieval: NDCG, MRR, and Recall
ingested-papers        5  Forgetting Mechanisms in Neural Memory Systems
ingested-papers       15  Letta: Agent Memory Beyond the Context Window
ingested-papers        7  MemGPT: Towards LLMs as Operating Systems
ingested-papers        5  Query Expansion and Hypothetica


### 3a. The tenant boundary is real

Before going further, confirm the claim the rest of the notebook leans on. Switch the
session to a different tenant and the entire corpus disappears. Same query, same connection, no predicate changed by the application.

In [7]:

cur.callproc("set_memory_ctx.set_tenant", [TENANT_GLOBEX])
cur.execute("SELECT COUNT(*) FROM knowledge_base_chunk")
print("chunks visible as tenant_globex:", cur.fetchone()[0])

cur.callproc("set_memory_ctx.set_tenant", [TENANT_ACME])
cur.execute("SELECT COUNT(*) FROM knowledge_base_chunk")
print("chunks visible as tenant_acme: ", cur.fetchone()[0])

chunks visible as tenant_globex: 0
chunks visible as tenant_acme:  137



## 4. Stage 1: query understanding

Before anything touches an index, the raw question is decomposed into a small plan object
that supplies the parameters for every stage after it:

- `entities`: the exact-match identifiers (`Letta`, `AgentCore`) that the lexical pool
  will pin. Embedding models under-weight rare proper nouns, so these have to survive as
  literal strings.
- `expanded terms`: the concept vocabulary. Jane says *memory eviction*; the Letta paper
  says *recursive summarization*; the AgentCore docs say *expiry* and *retention*. Nothing
  retrieves the second source unless something bridges that gap.
- `intent`: a comparison, which tells Stage 6 to reserve budget for both sources rather
  than filling the window with whichever document embeds closer.
- `scope`: the collections and time bounds Stage 2 will filter on.

In production this is one fast-model call returning JSON. Here it is deterministic, so the
notebook runs with no API key and produces identical output every time.

The thesaurus needs one guardrail. If the expansion map
contained the entry *"memory eviction" → "event expiry"*, this notebook would be answering
Jane's question in Stage 1 and letting retrieval take the credit. The map below is a general
memory-lifecycle vocabulary cluster, independent of any particular question. No entry is
keyed to an answer.
`eviction`, `expiry`, `retention`, `pruning` and `forgetting` belong together in that
cluster whether or not anyone ever asks Jane's question. A production system would generate
this from the model, or maintain it as a domain thesaurus.

In [8]:

# A small domain thesaurus over agent-memory lifecycle vocabulary. Deliberately organized
# by concept cluster rather than by question, so no entry encodes an answer.
CONCEPT_CLUSTERS = [
    {"eviction", "evict", "expiry", "expire", "retention", "pruning", "prune",
     "forgetting", "compaction", "truncation", "flush"},
    {"summarization", "summary", "summarize", "compression", "distillation", "rollup"},
    {"context window", "working set", "working context", "main context", "buffer"},
    {"long-term memory", "archival memory", "durable memory", "persistent memory"},
    {"short-term memory", "session memory", "recall memory", "event stream"},
    {"retrieval", "search", "recall", "lookup"},
]

# Tokens that look like proper nouns but are ordinary sentence words.
# Sentence-initial capitals are not entities. Without this filter "Which", "Why" and
# "Do" enter the lexical query as identifiers and match half the corpus.
_STOP_CAPS = {"what", "how", "the", "did", "and", "does", "that", "compare", "to", "a",
              "an", "when", "which", "where", "why", "is", "do", "can", "i", "in", "if",
              "should", "are", "was", "were", "will", "would", "could", "no", "not"}

# Words too common in this corpus to discriminate; used only by the fallback below.
_STOP_WORDS = _STOP_CAPS | {"of", "for", "on", "at", "by", "with", "from", "it", "its",
                            "be", "as", "or", "but", "so", "than", "then", "this", "these",
                            "there", "their", "you", "your", "we", "our", "they", "them",
                            "does", "did", "done", "use", "used", "using", "need", "needs",
                            "make", "makes", "get", "gets", "one", "any", "all", "some",
                            "ever", "much", "many", "most", "more", "less", "between"}

# Multi-word concepts must be matched before single words so "memory eviction" is not
# shattered into two unrelated lookups.
_PHRASES = sorted({t for c in CONCEPT_CLUSTERS for t in c if " " in t}, key=len, reverse=True)


def understand_query(question: str, *, collections=None, token_budget=2000) -> dict:
    """Decompose a question into the plan that parameterizes stages 2-6.

    Production swaps this for a single fast-model call returning the same JSON shape.
    """
    lowered = question.lower()

    # Entities: capitalized tokens that are not sentence-initial filler. Crude on purpose --
    # a model does this better, and the plan shape is what matters downstream.
    entities = []
    for tok in re.findall(r"\b[A-Z][A-Za-z0-9]+\b", question):
        if tok.lower() not in _STOP_CAPS and tok not in entities:
            entities.append(tok)

    # Concept expansion: for every cluster the question touches, take the whole cluster.
    matched, expanded = [], set()
    for phrase in _PHRASES:
        if phrase in lowered:
            matched.append(phrase)
    for cluster in CONCEPT_CLUSTERS:
        hit = any(p in lowered for p in cluster if " " in p) or any(
            re.search(rf"\b{re.escape(t)}\b", lowered) for t in cluster if " " not in t
        )
        if hit:
            expanded |= cluster

    # Intent: comparisons need budget reserved per source; lookups do not.
    if re.search(r"\bcompare\b|\bversus\b|\bvs\.?\b|difference between|how does that", lowered):
        intent = "comparison"
    elif re.search(r"\bhow (do|did) (we|i|you)\b|last time|previously", lowered):
        intent = "procedural_recall"
    else:
        intent = "factual_lookup"

    return {
        "question": question,
        "entities": entities,
        "expanded_terms": sorted(expanded),
        "intent": intent,
        "collections": collections or ["ingested-papers", "vendor-docs"],
        "token_budget": token_budget,
        "diversify_by": "document_id" if intent == "comparison" else None,
    }


def build_lexical_query(plan: dict) -> str:
    """Render the plan as an Oracle Text expression.

    Every term is wrapped in {} so Oracle Text treats it as a literal. This is not
    cosmetic: Oracle Text reserves ABOUT, ACCUM, AND, NEAR, NOT, MINUS, WITHIN and others
    as operators, so an ordinary English word like "within" is a syntax error unescaped
    (DRG-50901). Braces also neutralise hyphens and other punctuation.
    """
    def lit(t):
        return "{" + t.replace("{", "").replace("}", "") + "}"

    terms = [lit(e) for e in plan["entities"]]
    for t in plan["expanded_terms"]:
        terms.append(lit(t))

    # Fallback: a question with no proper nouns and no thesaurus hit would otherwise
    # produce an empty CONTAINS expression, which is a parser error rather than an empty
    # result. Fall back to the question's own content words.
    if not terms:
        words = [w for w in re.findall(r"[A-Za-z0-9]+", plan["question"].lower())
                 if w not in _STOP_WORDS and len(w) > 2]
        terms = [lit(w) for w in dict.fromkeys(words)] or [lit("memory")]

    return " OR ".join(terms)


QUESTION = ("What did the Letta paper say about memory eviction, and how does that "
            "compare to what the AgentCore docs recommend?")

plan = understand_query(QUESTION)
plan["lexical_query"] = build_lexical_query(plan)

print("intent:      ", plan["intent"])
print("entities:    ", plan["entities"])
print("collections: ", plan["collections"])
print("diversify_by:", plan["diversify_by"])
print("expanded terms:")
for t in plan["expanded_terms"]:
    print("   ", t)
print("\nlexical query for Oracle Text:")
print("   ", plan["lexical_query"])

intent:       comparison
entities:     ['Letta', 'AgentCore']
collections:  ['ingested-papers', 'vendor-docs']
diversify_by: document_id
expanded terms:
    compaction
    evict
    eviction
    expire
    expiry
    flush
    forgetting
    prune
    pruning
    retention
    truncation

lexical query for Oracle Text:
    {Letta} OR {AgentCore} OR {compaction} OR {evict} OR {eviction} OR {expire} OR {expiry} OR {flush} OR {forgetting} OR {prune} OR {pruning} OR {retention} OR {truncation}



The expansion matters here. `eviction` pulled in `expiry` and `retention`,
which is what gives the lexical pool any chance of reaching the AgentCore chunk, because that document never uses the word Jane typed. The entity list is what will pin `AgentCore`
itself, since the embedding treats a rare product name as a weak topical hint.

Skipping this stage has a measurable cost. With no expansion, the lexical pool searches
for `eviction` in a
corpus where the answering document says `expiry`, and returns nothing useful. The vector
pool is then the only signal, and §3's probe already showed where that lands the AgentCore
passage.


## 5. Stages 2, 3 and 4: one SQL statement

Metadata filtering, both candidate generators, and reciprocal rank fusion run as a single
query.

Read it in four parts:

- `Stage 2` is the `WHERE` clause, repeated inside each pool: collection restriction,
  lifecycle (`valid_until`, `deleted_at`), scope inheritance (`user_id IS NULL OR ...`),
  and the `section_type` exclusion. Repeating it is the point: each generator ranks only what the filter allows, which is what *filter early* means in practice.
- `Stage 3a` is `vec_pool`: a vector-distance candidate query ordered by cosine distance, ranked with
  `ROW_NUMBER()`.
- `Stage 3b` is `lex_pool`: an Oracle Text `CONTAINS` probe ordered by `SCORE()`.
- `Stage 4` is `fused`: a `FULL OUTER JOIN` between the two pools, with reciprocal rank
  fusion expressed as `1/(60 + rank)` summed across whichever lists a chunk appeared in.
  A chunk found by both generators collects both terms and rises; a chunk found by only
  one still competes.

`tenant_id` appears nowhere. RLS appends it to every reference of both tables, in both pools and on the final join, so there is no place to forget it.

The notebook carries `vec_rank` and `lex_rank` through the fusion so you can see the
disagreement between the generators. The article's version returns only `rrf_score`.

In [9]:

HYBRID_SQL = """
WITH
-- Stage 3a: vector candidates over the metadata-filtered set
vec_pool AS (
  SELECT c.id,
         ROW_NUMBER() OVER (
           ORDER BY VECTOR_DISTANCE(c.embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA),
             COSINE)) AS vec_rank
    FROM knowledge_base_chunk c
    JOIN knowledge_base_document d ON d.id = c.document_id
   WHERE d.collection IN ('ingested-papers', 'vendor-docs')     -- Stage 2 starts here
     AND d.valid_until IS NULL AND d.deleted_at IS NULL
     AND c.deleted_at IS NULL
     AND (d.user_id IS NULL OR d.user_id = :user_id)
     AND JSON_VALUE(c.metadata, '$.section_type') <> 'references'
   ORDER BY VECTOR_DISTANCE(c.embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA),
             COSINE)
   FETCH FIRST 50 ROWS ONLY
),
-- Stage 3b: lexical candidates over the same filtered set
lex_pool AS (
  -- DENSE_RANK keeps equal Oracle Text scores tied instead of treating arbitrary tie
  -- order as signal during reciprocal rank fusion.
  SELECT c.id,
         DENSE_RANK() OVER (ORDER BY SCORE(1) DESC) AS lex_rank
    FROM knowledge_base_chunk c
    JOIN knowledge_base_document d ON d.id = c.document_id
   WHERE CONTAINS(c.content, :lex_query, 1) > 0
     AND d.collection IN ('ingested-papers', 'vendor-docs')
     AND d.valid_until IS NULL AND d.deleted_at IS NULL
     AND c.deleted_at IS NULL
     AND (d.user_id IS NULL OR d.user_id = :user_id)
     AND JSON_VALUE(c.metadata, '$.section_type') <> 'references'
   ORDER BY SCORE(1) DESC
   FETCH FIRST 50 ROWS ONLY
),
-- Stage 4: reciprocal rank fusion across the two pools
fused AS (
  SELECT COALESCE(v.id, l.id) AS id,
         v.vec_rank, l.lex_rank,
         COALESCE(1 / (60 + v.vec_rank), 0)
       + COALESCE(1 / (60 + l.lex_rank), 0) AS rrf_score
    FROM vec_pool v
    FULL OUTER JOIN lex_pool l ON v.id = l.id
)
SELECT f.rrf_score, f.vec_rank, f.lex_rank,
       c.id, c.content, c.chunk_index, c.metadata,
       d.id AS document_id, d.title, d.source_uri, d.version
  FROM fused f
  JOIN knowledge_base_chunk c    ON c.id = f.id
  JOIN knowledge_base_document d ON d.id = c.document_id
 ORDER BY f.rrf_score DESC
 FETCH FIRST 40 ROWS ONLY
-- tenant_id predicate appended automatically by RLS on every table reference
"""

# oracledb maps Oracle's native JSON type straight to a Python dict, so metadata needs
# no parsing. Guard for str anyway in case fetch settings change.
def meta_of(row):
    m = row["metadata"]
    return json.loads(m) if isinstance(m, str) else (m or {})


t0 = time.perf_counter()
cur.execute(HYBRID_SQL, query=plan["question"], lex_query=plan["lexical_query"],
            user_id=USER_JANE)
rows = cur.fetchall()
elapsed = (time.perf_counter() - t0) * 1000

cols = [d[0].lower() for d in cur.description]
candidates = [dict(zip(cols, r)) for r in rows]

print(f"{len(candidates)} fused candidates in {elapsed:.0f} ms\n")
print(f"{'#':>2}  {'rrf':>7}  {'vec':>4} {'lex':>4}  {'source':<34} section")
print("-" * 92)
for i, r in enumerate(candidates[:15], 1):
    v = str(r["vec_rank"]) if r["vec_rank"] is not None else "-"
    l = str(r["lex_rank"]) if r["lex_rank"] is not None else "-"
    section = meta_of(r).get("section_type", "?")
    print(f"{i:>2}  {r['rrf_score']:.5f}  {v:>4} {l:>4}  {r['title'][:34]:<34} {section}")

40 fused candidates in 94 ms

 #      rrf   vec  lex  source                             section
--------------------------------------------------------------------------------------------
 1  0.03202     1    4  Letta: Agent Memory Beyond the Con memory-management
 2  0.03175     2    4  Letta: Agent Memory Beyond the Con architecture
 3  0.03150     3    4  In-Memory Session State for LLM Ap methods
 4  0.03125     4    4  Letta: Agent Memory Beyond the Con memory-management
 5  0.03102     6    3  Forgetting Mechanisms in Neural Me discussion
 6  0.03101     5    4  MemGPT: Towards LLMs as Operating  memory-management
 7  0.03089     9    1  Letta: Agent Memory Beyond the Con memory-management
 8  0.03083     8    2  Amazon Bedrock AgentCore Memory De integration
 9  0.03055     7    4  Letta: Agent Memory Beyond the Con memory-management
10  0.03041    10    2  Amazon Bedrock AgentCore Memory De overview
11  0.02976    12    3  Amazon Bedrock AgentCore Memory De retention
12  0.02


Read the `vec` and `lex` columns rather than the ranking. The chunks at the top are
mostly the ones that appeared in **both** pools. A chunk found twice collects two
reciprocal terms and outranks a chunk that dominated one list alone. That is the entire
mechanism, and it is four lines of SQL.

Note also what is absent: the references chunk that was lexical rank 1 with a score of 69
does not appear at all. Stage 2 removed it before either generator ran.

In [10]:

# Where did the two chunks that actually answer Jane's question land?
def find_rank(cands, title_prefix, section):
    for i, r in enumerate(cands, 1):
        meta = meta_of(r)
        if r["title"].startswith(title_prefix) and meta.get("section_type") == section:
            return i, r
    return None, None

print("Answer chunks after fusion (was: vector rank 1 and rank 21):\n")
for label, prefix, section in [("Letta   ", "Letta", "memory-management"),
                               ("AgentCore", "Amazon", "retention")]:
    rank, row = find_rank(candidates, prefix, section)
    if row:
        print(f"  {label}  fused rank {rank:>2}  (vec {row['vec_rank']}, lex {row['lex_rank']})")
        print(f"             {row['content'][:78]}...")

# How redundant is the top of the list? This is what reranking and budgeting must fix.
from collections import Counter
top10 = Counter(r["title"][:38] for r in candidates[:10])
print("\nDocument spread in fused top-10:")
for title, n in top10.most_common():
    print(f"  {n} x {title}")

Answer chunks after fusion (was: vector rank 1 and rank 21):

  Letta     fused rank  1  (vec 1, lex 4)
             Eviction in Letta is driven by a pressure signal on the working context. As th...
  AgentCore  fused rank 11  (vec 12, lex 3)
             Retention is configured per memory resource rather than per session. The setti...

Document spread in fused top-10:
  5 x Letta: Agent Memory Beyond the Context
  2 x Amazon Bedrock AgentCore Memory Develo
  1 x In-Memory Session State for LLM Applic
  1 x Forgetting Mechanisms in Neural Memory
  1 x MemGPT: Towards LLMs as Operating Syst



### 5a. The execution plan

The interesting part of the plan is not any single operation. It is *who chose the order*.
The optimizer decides whether the collection filter runs before or after the vector index
probe based on statistics about how selective each predicate actually is. In a hand-built
pipeline that decision is hardcoded by whoever wrote the orchestration code, using whatever
they believed about the data the week they wrote it.

In [11]:

cur.execute("EXPLAIN PLAN FOR " + HYBRID_SQL,
            query=plan["question"], lex_query=plan["lexical_query"], user_id=USER_JANE)
# DBMS_XPLAN renders the tree, row estimates and predicates. Reading the raw plan_table
# with a CONNECT BY works too, but statement_id is NULL here and would break the join.
for row in cur.execute("SELECT plan_table_output FROM TABLE(DBMS_XPLAN.DISPLAY())"):
    print(row[0])

Plan hash value: 3766875347
 
-----------------------------------------------------------------------------------------------------------------------------
| Id  | Operation                                         | Name                    | Rows  | Bytes | Cost (%CPU)| Time     |
-----------------------------------------------------------------------------------------------------------------------------
|   0 | SELECT STATEMENT                                  |                         |     1 |  7009 |     9  (34)| 00:00:01 |
|*  1 |  COUNT STOPKEY                                    |                         |       |       |            |          |
|   2 |   VIEW                                            |                         |     1 |  7009 |     9  (34)| 00:00:01 |
|*  3 |    SORT ORDER BY STOPKEY                          |                         |     1 |  7166 |     9  (34)| 00:00:01 |
|   4 |     NESTED LOOPS                                  |                         |   


### 5b. One statement, one snapshot

The whole statement reads a single transactional snapshot. The vector pool, the lexical
pool and the final join all see the
same instant of the database. If a document were being superseded mid-query by a Promote
step on another connection, this query would see it entirely old or entirely new, never a chunk from each.

The polyglot equivalent has no shared snapshot. A document superseded between the vector
call and the lexical call contributes its old chunks to one pool and its new chunks to the
other, and the fusion silently mixes two versions of the same document.

In [12]:

# A single statement reads a single snapshot. Both correlated subqueries below resolve
# against the same instant, exactly as vec_pool and lex_pool do inside the hybrid query.
cur.execute("""
SELECT (SELECT COUNT(*) FROM knowledge_base_document) AS docs,
       (SELECT COUNT(*) FROM knowledge_base_chunk)    AS chunks,
       (SELECT COUNT(*) FROM knowledge_base_chunk WHERE embedding IS NOT NULL) AS embedded
  FROM dual
""")
ndocs, nchunks, nembedded = cur.fetchone()
print(f"one statement, one snapshot: {ndocs} documents, {nchunks} chunks, {nembedded} embedded")
print("vec_pool, lex_pool and the final join all resolve against that same instant --")
print("a document superseded mid-query is seen entirely old or entirely new, never both.")

one statement, one snapshot: 23 documents, 137 chunks, 137 embedded
vec_pool, lex_pool and the final join all resolve against that same instant --
a document superseded mid-query is seen entirely old or entirely new, never both.



## 6. Stage 5: reranking, in the database

Everything so far used **bi-encoders**: the query and every chunk were embedded
separately, at different times, with no knowledge of each other, and relevance was
approximated by the distance between those independent points. That is what makes
candidate generation cheap, and it is also the ceiling on its precision.

A **cross-encoder** scores the pair together. Query and candidate go through the model in
one pass, attention flows between their tokens, and the output is a relevance score
computed with both texts available. It can tell that a
chunk mentioning *message eviction policy* answers a question about *memory eviction*, and
that a chunk merely containing the word "Letta" in a citation list answers nothing. This is
also why a cross-encoder cannot generate candidates: scoring every chunk against every
query is a full scan of the most expensive kind.

Here it runs **inside the database**, which matters for more than tidiness. The candidates
are tenant data. A sidecar reranker moves that text outside the boundary RLS enforces, and
a hosted reranking API moves it outside your infrastructure entirely.

### Producing the model

Oracle ships no prebuilt reranker, so the ONNX model is built once with OML4Py's
`ONNXPipeline` and loaded like any other model. `function=REGRESSION` is the giveaway that
a cross-encoder is, in database terms, a two-column regression: two texts in, one score out.

```python
# one-time, in an OML4Py 2.1 environment (Linux x86-64)
from oml import MiningFunction, ONNXPipeline
ONNXPipeline('BAAI/bge-reranker-base',
             function=MiningFunction.REGRESSION).export2file('bge_reranker_base')
```

```sql
BEGIN
  DBMS_VECTOR.LOAD_ONNX_MODEL('ONNX_DIR', 'bge_reranker_base.onnx', 'BGE_RERANKER',
    JSON('{"function":"regression","regressionOutput":"output"}'));
END;
```

Do not add an `input` mapping here, even though Oracle's reranking-pipeline example
shows one (`"input":{"first_input":["DATA1"],"second_input":["DATA2"]}`). That clause
renames the model's ONNX tensors to mining attributes, which is what you want when scoring
*table columns* in bulk. The rerank APIs bind the query and documents to the model's
**native** tensor names, so a model loaded with that mapping fails with
`ORA-54421: Missing mining attribute: DATA1`. The error names the attribute the model wants; the caller supplied different ones. Omit the clause and both interfaces work.

Two constraints appeared in the tested OML4Py 2.1.1 and Oracle AI Database 26ai
environment. The XLM-RoBERTa rerankers exported successfully, while the tested
BERT-based `cross-encoder/ms-marco-*` model failed during tokenizer validation.
`BAAI/bge-reranker-v2-m3` exported to a 2.2 GB artifact, then failed to load; quantization
was also refused for the model over 2 GB. These are observed, release-specific results,
so recheck model compatibility and size handling against the versions you deploy.

In [13]:

cur.execute("""
SELECT model_name, mining_function, ROUND(model_size/1024/1024, 1) AS mb
  FROM user_mining_models ORDER BY model_name
""")
print(f"{'model':<22} {'function':<12} {'MB':>8}")
for name, fn, mb in cur:
    print(f"{name:<22} {fn:<12} {mb:>8}")

cur.execute("""
SELECT attribute_name, attribute_type FROM user_mining_model_attributes
 WHERE model_name = 'BGE_RERANKER' ORDER BY attribute_name
""")
print("\nBGE_RERANKER attributes:", [r[0] for r in cur])

# Warm the model. The first PREDICTION pays a one-time load; charging that to per-query
# latency is an easy way to publish a wrong number.
t0 = time.perf_counter()
cur.execute("SELECT PREDICTION(BGE_RERANKER USING 'a' AS FIRST_INPUT, 'b' AS SECOND_INPUT) FROM dual")
cur.fetchone()
print(f"\ncold first call: {(time.perf_counter()-t0)*1000:.0f} ms (model load, paid once)")

model                  function           MB
ALL_MINILM_L12_V2      EMBEDDING       127.1
BGE_RERANKER           REGRESSION      275.5

BGE_RERANKER attributes: ['FIRST_INPUT', 'ORA$ONNXTARGET', 'SECOND_INPUT']

cold first call: 853 ms (model load, paid once)



cold first call: 1107 ms (model load, paid once)



Two interfaces reach the same model. `DBMS_VECTOR_CHAIN.UTL_TO_RERANK` is the documented
entry point: hand it a query and a JSON array of documents and it returns them reordered
with scores. `PREDICTION` exposes the same model as a SQL expression, which is what lets
Stage 5 fuse into the retrieval statement instead of becoming a second round trip.

In [14]:

DOCS_PROBE = [
    "Letta frames eviction as recursive summarization; the oldest span of messages is "
    "compressed into a summary and the raw messages are evicted from the window.",
    "Raising target accuracy increases the number of neighbors visited, improving recall "
    "at the cost of query time.",
    "Each memory resource defines an event expiry period, specified in days.",
]
RQ = "What did the Letta paper say about memory eviction?"

cur.execute("""
SELECT DBMS_VECTOR_CHAIN.UTL_TO_RERANK(:q, JSON(:docs), JSON(:params)) FROM dual
""", q=RQ,
     docs=json.dumps({"documents": DOCS_PROBE}),
     params=json.dumps({"provider": "database", "model": "BGE_RERANKER",
                        "return_docs": True, "top_n": 3}))
out = cur.fetchone()[0]
ranked = json.loads(out) if isinstance(out, str) else out

print("UTL_TO_RERANK returned documents in relevance order:")
for r in ranked:
    print(f"  score {float(r['score']):>8.3f}  (input #{int(r['index'])})  {r['content'][:58]}...")

cur.execute("""
SELECT PREDICTION(BGE_RERANKER USING :q AS FIRST_INPUT, :d AS SECOND_INPUT) FROM dual
""", q=RQ, d=DOCS_PROBE[0])
print(f"\nPREDICTION on the same pair: {float(cur.fetchone()[0]):.3f} -- identical score,")
print("reached through a SQL expression instead of a function call.")

UTL_TO_RERANK returned documents in relevance order:
  score   -2.211  (input #0)  Letta frames eviction as recursive summarization; the olde...
  score   -9.526  (input #2)  Each memory resource defines an event expiry period, speci...
  score  -10.177  (input #1)  Raising target accuracy increases the number of neighbors ...

PREDICTION on the same pair: -2.211 -- identical score,
reached through a SQL expression instead of a function call.



### 6a. Rerank the chunk, or the chunk plus its title?

Scoring the query against `c.content` alone performs
**worse than not reranking at all**, and the reason is worth internalizing: these chunks
average 25 words. A cross-encoder handed a bare sentence has no idea which document it came
from, and these passages are far shorter than the ones such models are trained on.

Prepending the document title restores that context for the cost of a string concatenation.

In [15]:

PROBE = "How is tenant isolation enforced at the database level?"

RANK_PROBE = """
SELECT title, st, rk FROM (
  SELECT d.title, JSON_VALUE(c.metadata,'$.section_type') AS st,
         ROW_NUMBER() OVER (ORDER BY PREDICTION(
             BGE_RERANKER USING :q AS FIRST_INPUT, {expr} AS SECOND_INPUT) DESC) AS rk
    FROM knowledge_base_chunk c
    JOIN knowledge_base_document d ON d.id = c.document_id
   WHERE JSON_VALUE(c.metadata,'$.section_type') <> 'references')
 WHERE title LIKE 'Multi-Tenant%'
 ORDER BY rk FETCH FIRST 3 ROWS ONLY
"""

print(f"query: {PROBE!r}\n")
for label, expr in [("chunk only    ", "c.content"),
                    ("title + chunk ", "d.title || '. ' || c.content")]:
    cur.execute(RANK_PROBE.format(expr=expr), q=PROBE)
    ranks = [f"{st}@{rk}" for _, st, rk in cur]
    print(f"  {label} -> best Multi-Tenant chunks at {ranks}")

print("\nSame model, same query, same candidates. Only the text handed to the")
print("cross-encoder changed.")

query: 'How is tenant isolation enforced at the database level?'

  chunk only     -> best Multi-Tenant chunks at ['methods@11', 'discussion@21', 'methods@24']
  title + chunk  -> best Multi-Tenant chunks at ['discussion@1', 'methods@2', 'methods@3']

Same model, same query, same candidates. Only the text handed to the
cross-encoder changed.


  chunk only     -> best Multi-Tenant chunks at ['methods@11', 'discussion@21', 'methods@24']


  title + chunk  -> best Multi-Tenant chunks at ['discussion@1', 'methods@2', 'methods@3']

Same model, same query, same candidates. Only the text handed to the
cross-encoder changed.



### 6b. Stage 5 as a column expression

The registered model makes reranking a `PREDICTION` in the select list. Stages 2 through
5 remain in a single statement: filter,
generate candidates two ways, fuse the ranks, and rescore the survivors with a cross-encoder,
all in one plan against one snapshot.

In [16]:

RERANK_SQL = """
WITH vec_pool AS (
  SELECT c.id, ROW_NUMBER() OVER (
           ORDER BY VECTOR_DISTANCE(c.embedding,
             VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)) AS vec_rank
    FROM knowledge_base_chunk c
    JOIN knowledge_base_document d ON d.id = c.document_id
   WHERE d.collection IN ('ingested-papers','vendor-docs')
     AND d.valid_until IS NULL AND d.deleted_at IS NULL AND c.deleted_at IS NULL
     AND (d.user_id IS NULL OR d.user_id = :user_id)
     AND JSON_VALUE(c.metadata, '$.section_type') <> 'references'
   ORDER BY VECTOR_DISTANCE(c.embedding,
            VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)
   FETCH FIRST 50 ROWS ONLY),
lex_pool AS (
  -- DENSE_RANK, not ROW_NUMBER: Oracle Text SCORE() is a coarse integer and ties are
  -- common on short chunks. ROW_NUMBER would order tied chunks arbitrarily and RRF would
  -- read that arbitrary order as signal.
  SELECT c.id, DENSE_RANK() OVER (ORDER BY SCORE(1) DESC) AS lex_rank
    FROM knowledge_base_chunk c
    JOIN knowledge_base_document d ON d.id = c.document_id
   WHERE CONTAINS(c.content, :lex_query, 1) > 0
     AND d.collection IN ('ingested-papers','vendor-docs')
     AND d.valid_until IS NULL AND d.deleted_at IS NULL AND c.deleted_at IS NULL
     AND (d.user_id IS NULL OR d.user_id = :user_id)
     AND JSON_VALUE(c.metadata, '$.section_type') <> 'references'
   ORDER BY SCORE(1) DESC
   FETCH FIRST 50 ROWS ONLY),
fused AS (
  SELECT COALESCE(v.id, l.id) AS id,
         COALESCE(1/(60 + v.vec_rank), 0) + COALESCE(1/(60 + l.lex_rank), 0) AS rrf_score
    FROM vec_pool v FULL OUTER JOIN lex_pool l ON v.id = l.id),
candidates AS (
  SELECT id, rrf_score FROM fused ORDER BY rrf_score DESC FETCH FIRST 40 ROWS ONLY)
SELECT c.id, c.content, c.metadata, d.id AS document_id, d.title, d.source_uri, d.version,
       k.rrf_score,
       PREDICTION(BGE_RERANKER USING :query AS FIRST_INPUT,
                  d.title || '. ' || c.content AS SECOND_INPUT) AS rerank_score
  FROM candidates k
  JOIN knowledge_base_chunk c    ON c.id = k.id
  JOIN knowledge_base_document d ON d.id = c.document_id
 ORDER BY rerank_score DESC
-- tenant_id appended by RLS on every table reference
"""

t0 = time.perf_counter()
cur.execute(RERANK_SQL, query=plan["question"], lex_query=plan["lexical_query"],
            user_id=USER_JANE)
cols = [d[0].lower() for d in cur.description]
reranked = [dict(zip(cols, r)) for r in cur.fetchall()]
rerank_ms = (time.perf_counter() - t0) * 1000

print(f"reranked {len(reranked)} candidates in {rerank_ms:.0f} ms "
      f"({rerank_ms/max(len(reranked),1):.0f} ms/pair)\n")
print(f"{'#':>2}  {'rerank':>8}  {'rrf':>8}  {'source':<36} section")
print("-" * 82)
for i, r in enumerate(reranked[:10], 1):
    print(f"{i:>2}  {r['rerank_score']:>8.3f}  {r['rrf_score']:>8.5f}  "
          f"{r['title'][:36]:<36} {meta_of(r).get('section_type','?')}")

reranked 40 candidates in 3210 ms (80 ms/pair)

 #    rerank       rrf  source                               section
----------------------------------------------------------------------------------
 1    -0.036   0.03125  Letta: Agent Memory Beyond the Conte memory-management
 2    -0.762   0.03202  Letta: Agent Memory Beyond the Conte memory-management
 3    -2.933   0.01562  Letta: Agent Memory Beyond the Conte evaluation
 4    -3.281   0.03055  Letta: Agent Memory Beyond the Conte memory-management
 5    -3.716   0.02837  Amazon Bedrock AgentCore Memory Deve retention
 6    -3.779   0.01282  Amazon Bedrock AgentCore Memory Deve strategies
 7    -3.864   0.03041  Amazon Bedrock AgentCore Memory Deve overview
 8    -3.943   0.01149  Amazon Bedrock AgentCore Memory Deve namespaces
 9    -4.148   0.03083  Amazon Bedrock AgentCore Memory Deve integration
10    -4.212   0.03089  Letta: Agent Memory Beyond the Conte memory-management



Compare that ordering against §5. Fusion put a near-duplicate Letta chunk on top and left
the canonical eviction passage further down; the cross-encoder, seeing query and chunk
together, corrects it. That reordering is the entire value of Stage 5. It cannot add anything the candidate set lacks, which is why recall is Stage 3's job and ordering is
Stage 5's.


## 7. Stage 6: context budgeting

The token budget is fixed before retrieval starts, and the reranked list gets cut to fit.
Cutting is a knapsack decision rather than a truncation:

- Diversity across sources for a comparison question. Jane asked how two documents
  compare; a window holding six Letta chunks and no AgentCore chunk cannot answer that,
  no matter how well each chunk scored.
- Drop near-duplicates. Four chunks saying the same thing about eviction cost four
  chunks of budget and add one chunk of information.
- Stop at the score cliff. When relevance falls off sharply, more tokens of weak
  context is just more expensive weak context.

What you leave out is a decision too. The output is the **working set**: the ranked,
budgeted, provenance-tagged slice of memory the agent's Infer & Act step consumes.

In [17]:

def approx_tokens(text: str) -> int:
    """Cheap proxy: ~4 characters per token. Swap in a real tokenizer in production."""
    return max(1, len(text) // 4)


def jaccard(a: str, b: str) -> float:
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa | sb else 0.0


def budget_context(candidates, *, max_tokens=1200, diversify_by=None,
                   per_source_cap=3, dup_threshold=0.6, cliff_ratio=0.45):
    """Cut a reranked list to a token budget under diversity and redundancy constraints."""
    working, used, per_source = [], 0, {}
    top = candidates[0]["rerank_score"] if candidates else 0.0
    # Scores are logits and can be negative, so measure the cliff on the observed range.
    lo = min(c["rerank_score"] for c in candidates) if candidates else 0.0
    span = (top - lo) or 1.0

    for cand in candidates:
        # Score cliff: stop once relevance has fallen most of the way toward the worst
        # candidate in the set. Measured on the observed range because these are logits.
        rel = (cand["rerank_score"] - lo) / span
        if working and rel < cliff_ratio:
            break

        key = cand[diversify_by] if diversify_by else None
        if key is not None and per_source.get(key, 0) >= per_source_cap:
            continue

        if any(jaccard(cand["content"], w["content"]) >= dup_threshold for w in working):
            continue

        cost = approx_tokens(cand["content"])
        if used + cost > max_tokens:
            continue
        working.append(cand)
        used += cost
        if key is not None:
            per_source[key] = per_source.get(key, 0) + 1

    return working, used


working_set, tokens_used = budget_context(
    reranked, max_tokens=plan["token_budget"], diversify_by=plan["diversify_by"])

print(f"working set: {len(working_set)} chunks, ~{tokens_used} tokens "
      f"(budget {plan['token_budget']}), intent={plan['intent']}\n")
for i, r in enumerate(working_set, 1):
    print(f"{i:>2}. [{r['rerank_score']:>7.3f}] {r['title'][:44]:<44} "
          f"{meta_of(r).get('section_type','?')}")

from collections import Counter
print("\nsources represented:", dict(Counter(r["title"][:34] for r in working_set)))
print("\nprovenance on every row (what makes citation possible):")
for r in working_set[:2]:
    print(f"  document_id={r['document_id']}  version={r['version']}")
    print(f"    {r['source_uri']}")

working set: 7 chunks, ~479 tokens (budget 2000), intent=comparison

 1. [ -0.036] Letta: Agent Memory Beyond the Context Windo memory-management
 2. [ -0.762] Letta: Agent Memory Beyond the Context Windo memory-management
 3. [ -2.933] Letta: Agent Memory Beyond the Context Windo evaluation
 4. [ -3.716] Amazon Bedrock AgentCore Memory Developer Gu retention
 5. [ -3.779] Amazon Bedrock AgentCore Memory Developer Gu strategies
 6. [ -3.864] Amazon Bedrock AgentCore Memory Developer Gu overview
 7. [ -5.292] Forgetting Mechanisms in Neural Memory Syste discussion

sources represented: {'Letta: Agent Memory Beyond the Con': 3, 'Amazon Bedrock AgentCore Memory De': 3, 'Forgetting Mechanisms in Neural Me': 1}

provenance on every row (what makes citation possible):
  document_id=kbd_a1d89dc3adb9  version=1
    https://arxiv.org/abs/2310.08560
  document_id=kbd_a1d89dc3adb9  version=1
    https://arxiv.org/abs/2310.08560



## 8. The experiment from the top of the article

Hold the model fixed. Run Jane's question through three retrieval strategies and look at
what each one puts in the window. The model never changes; only retrieval does.

In [18]:

STRAT_SQL = {}

STRAT_SQL["1. Pure vector top-k"] = """
SELECT d.title, JSON_VALUE(c.metadata,'$.section_type') AS st, c.content
  FROM knowledge_base_chunk c JOIN knowledge_base_document d ON d.id = c.document_id
 WHERE d.collection IN ('ingested-papers','vendor-docs')
   AND d.valid_until IS NULL AND d.deleted_at IS NULL AND c.deleted_at IS NULL
 ORDER BY VECTOR_DISTANCE(c.embedding,
          VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)
 FETCH FIRST 6 ROWS ONLY
"""

def run_strategy(name):
    if name.startswith("1."):
        cur.execute(STRAT_SQL[name], query=plan["question"])
        return [(t, s, c) for t, s, c in cur]
    if name.startswith("2."):
        cur.execute(HYBRID_SQL, query=plan["question"],
                    lex_query=plan["lexical_query"], user_id=USER_JANE)
        cols_ = [d[0].lower() for d in cur.description]
        rows_ = [dict(zip(cols_, r)) for r in cur.fetchall()][:6]
        return [(r["title"], meta_of(r).get("section_type","?"), r["content"]) for r in rows_]
    return [(r["title"], meta_of(r).get("section_type","?"), r["content"])
            for r in reranked[:6]]

for name in ["1. Pure vector top-k", "2. Hybrid (RRF)", "3. Hybrid + rerank"]:
    print(f"\n=== {name} ===")
    for i, (title, st, content) in enumerate(run_strategy(name), 1):
        mark = "**" if (title.startswith("Letta") and st == "memory-management") or \
                       (title.startswith("Amazon") and st == "retention") else "  "
        print(f" {mark}{i}. {title[:40]:<40} [{st}]")
print("\n** marks a chunk that actually answers one half of Jane's question.")


=== 1. Pure vector top-k ===
 **1. Letta: Agent Memory Beyond the Context W [memory-management]
   2. Letta: Agent Memory Beyond the Context W [architecture]
   3. In-Memory Session State for LLM Applicat [methods]
 **4. Letta: Agent Memory Beyond the Context W [memory-management]
   5. MemGPT: Towards LLMs as Operating System [memory-management]
   6. Forgetting Mechanisms in Neural Memory S [discussion]

=== 2. Hybrid (RRF) ===
 **1. Letta: Agent Memory Beyond the Context W [memory-management]
   2. Letta: Agent Memory Beyond the Context W [architecture]
   3. In-Memory Session State for LLM Applicat [methods]
 **4. Letta: Agent Memory Beyond the Context W [memory-management]
   5. Forgetting Mechanisms in Neural Memory S [discussion]
   6. MemGPT: Towards LLMs as Operating System [memory-management]

=== 3. Hybrid + rerank ===
 **1. Letta: Agent Memory Beyond the Context W [memory-management]
 **2. Letta: Agent Memory Beyond the Context W [memory-management]
   3. Letta: Agent Memo


### 8a. Does the working set let the agent answer?

Jane asked a comparison. The test is not whether good chunks appeared, but whether **both
sources** are represented well enough to compare.

In [19]:

def coverage(rows):
    letta = any(t.startswith("Letta") and s == "memory-management" for t, s, _ in rows)
    agent = any(t.startswith("Amazon") and s == "retention" for t, s, _ in rows)
    docs = len({t for t, _, _ in rows})
    return letta, agent, docs

print(f"{'strategy':<24} {'Letta answer':>13} {'AgentCore answer':>18} {'distinct docs':>14}")
print("-" * 73)
for name in ["1. Pure vector top-k", "2. Hybrid (RRF)", "3. Hybrid + rerank"]:
    l, a, d = coverage(run_strategy(name))
    print(f"{name:<24} {str(l):>13} {str(a):>18} {d:>14}")

wl, wa, wd = coverage([(r["title"], meta_of(r).get("section_type","?"), r["content"])
                       for r in working_set])
print(f"{'4. + budgeting (§7)':<24} {str(wl):>13} {str(wa):>18} {wd:>14}")

strategy                  Letta answer   AgentCore answer  distinct docs
-------------------------------------------------------------------------
1. Pure vector top-k              True              False              4
2. Hybrid (RRF)                   True              False              4
3. Hybrid + rerank                True               True              2
4. + budgeting (§7)               True               True              3


1. Pure vector top-k              True              False              4


2. Hybrid (RRF)                   True              False              4
3. Hybrid + rerank                True               True              2
4. + budgeting (§7)               True               True              3



## 9. Evaluating retrieval quality

Unmeasured retrieval is unmanaged retrieval. Everything above is an anecdote until it is
scored against labelled data, and anecdotes have terrible recall.

The metrics and when each earns its keep:

- `Recall@k`: of the truly relevant items, how many made the top k. The
  candidate-generation metric. If recall is bad, nothing downstream can save you.
- `NDCG@10`: rewards putting the most relevant items highest, with graded relevance,
  over roughly the number of results that fit a context budget. For agent memory retrieval
  this is the most informative single number, because ordering is what the budget stage
  consumes.
- `MRR`: how high the first relevant item lands. Right when one good hit is enough.

The golden set below is hand-authored because this notebook has no production traffic. In
a real system you harvest it from `conversation_memory`: every question, the candidate ids
the pipeline returned (logged as a `retrieval_result` event), and the downstream signal (the follow-up, the thumbs-down, the correction) is already in the trace, scoped and timestamped. Production traffic is the test set you already have.

In [20]:
GOLDEN_SET = [
    # ---------- the article's running question and its neighbourhood ----------
    ("What did the Letta paper say about memory eviction, and how does that compare to "
     "what the AgentCore docs recommend?",
     {("Letta", "memory-management"): 3, ("Amazon", "retention"): 3,
      ("Letta", "architecture"): 1, ("Letta", "abstract"): 1,
      ("Amazon", "overview"): 1, ("Amazon", "strategies"): 1}),

    ("How does Letta decide what to remove from the context window?",
     {("Letta", "memory-management"): 3, ("Letta", "architecture"): 1,
      ("MemGPT", "memory-management"): 1}),

    ("How long does AgentCore keep raw session events?",
     {("Amazon", "retention"): 3, ("Amazon", "short-term"): 1, ("Amazon", "overview"): 1}),

    ("What happens to long-term memories when the source events expire?",
     {("Amazon", "retention"): 3, ("Amazon", "strategies"): 2}),

    ("What is recursive summarization?",
     {("Letta", "memory-management"): 3, ("MemGPT", "memory-management"): 2,
      ("Conversation Summarization", "methods"): 2}),

    ("Which memory tiers does Letta use?",
     {("Letta", "architecture"): 3, ("Letta", "abstract"): 1,
      ("MemGPT", "architecture"): 1}),

    ("How do I configure what an agent remembers across sessions in AgentCore?",
     {("Amazon", "strategies"): 3, ("Amazon", "namespaces"): 2, ("Amazon", "retention"): 2}),

    ("How are AgentCore memories scoped to a particular user?",
     {("Amazon", "namespaces"): 3, ("Amazon", "overview"): 1}),

    ("What is the downside of summarizing conversation history?",
     {("Letta", "discussion"): 3, ("Conversation Summarization", "discussion"): 3,
      ("Conversation Summarization", "methods"): 1}),

    # ---------- context management and memory literature ----------
    ("What are the trade-offs between truncation, summarization and retrieval for long "
     "conversations?",
     {("Context Window Management", "methods"): 3, ("Context Window Management", "discussion"): 2,
      ("Context Window Management", "abstract"): 1}),

    ("How does MemGPT decide when to page information out of context?",
     {("MemGPT", "memory-management"): 3, ("MemGPT", "abstract"): 1,
      ("MemGPT", "architecture"): 1}),

    ("Is forgetting ever desirable in a memory system?",
     {("Forgetting Mechanisms", "abstract"): 3, ("Forgetting Mechanisms", "discussion"): 3,
      ("Forgetting Mechanisms", "methods"): 2, ("Forgetting Mechanisms", "evaluation"): 2}),

    ("How should conflicting facts be resolved in long-term memory?",
     {("Scalable Long-Term Memory", "methods"): 3, ("Forgetting Mechanisms", "methods"): 2}),

    ("What limits the usefulness of an extracted-fact memory layer?",
     {("Scalable Long-Term Memory", "discussion"): 3, ("Scalable Long-Term Memory", "evaluation"): 1}),

    ("Does a longer context window remove the need for retrieval?",
     {("Sliding Window Attention", "discussion"): 3, ("Sliding Window Attention", "evaluation"): 2,
      ("Context Window Management", "discussion"): 1}),

    ("Where do models attend least within a long context?",
     {("Sliding Window Attention", "evaluation"): 3, ("Sliding Window Attention", "discussion"): 1}),

    ("How are episodes segmented and consolidated in episodic memory?",
     {("Episodic Memory", "methods"): 3, ("Episodic Memory", "abstract"): 1,
      ("Episodic Memory", "discussion"): 1}),

    ("How should a dialogue system build its retrieval query when the user's turn is "
     "elliptical?",
     {("Retrieval-Augmented Dialogue", "methods"): 3, ("Query Expansion", "methods"): 2,
      ("Retrieval-Augmented Dialogue", "abstract"): 1}),

    # ---------- retrieval mechanics ----------
    ("When does lexical search beat dense retrieval?",
     {("BM25", "evaluation"): 3, ("BM25", "discussion"): 3, ("BM25", "abstract"): 2,
      ("Dense Passage Retrieval", "evaluation"): 2}),

    ("Why do dense retrievers struggle with rare identifiers and exact strings?",
     {("Dense Passage Retrieval", "evaluation"): 3, ("Dense Passage Retrieval", "discussion"): 3,
      ("BM25", "abstract"): 2, ("BM25", "methods"): 1}),

    ("Why can a cross-encoder not be used to search the whole corpus?",
     {("Cross-Encoder Reranking", "methods"): 3, ("Cross-Encoder Reranking", "abstract"): 2,
      ("Dense Passage Retrieval", "discussion"): 1}),

    ("Why does reranking improve ordering but not recall?",
     {("Cross-Encoder Reranking", "evaluation"): 3, ("Cross-Encoder Reranking", "discussion"): 3,
      ("Cross-Encoder Reranking", "methods"): 1}),

    ("How should sparse and dense retrieval scores be combined?",
     {("A Survey of Retrieval", "retrieval"): 3, ("BM25", "discussion"): 1}),

    ("Which metric should I use when the consumer reads results in order?",
     {("Evaluating Retrieval", "methods"): 3, ("Evaluating Retrieval", "discussion"): 2}),

    ("When is mean reciprocal rank the right metric?",
     {("Evaluating Retrieval", "methods"): 3, ("Evaluating Retrieval", "discussion"): 1}),

    ("How large should chunks be, and does overlap help?",
     {("Chunking Strategies", "methods"): 3, ("Chunking Strategies", "evaluation"): 3,
      ("Chunking Strategies", "discussion"): 2, ("Chunking Strategies", "abstract"): 1}),

    ("Does query expansion ever hurt retrieval quality?",
     {("Query Expansion", "evaluation"): 3, ("Query Expansion", "discussion"): 2,
      ("Query Expansion", "methods"): 1}),

    # ---------- infrastructure and vendor docs ----------
    ("How is tenant isolation enforced at the database level?",
     {("Multi-Tenant", "methods"): 3, ("Multi-Tenant", "discussion"): 2,
      ("Multi-Tenant", "overview"): 1, ("Namespace and Metadata", "discussion"): 1}),

    ("What does a metadata filter do that a similarity score cannot?",
     {("Namespace and Metadata", "methods"): 3, ("Namespace and Metadata", "discussion"): 2,
      ("Multi-Tenant", "methods"): 1, ("Namespace and Metadata", "overview"): 1}),

    ("How do I trade recall against query latency in a vector index?",
     {("Vector Index Tuning", "parameters"): 3, ("Vector Index Tuning", "overview"): 2,
      ("Vector Index Tuning", "discussion"): 1}),

    ("Do I need to rebuild the vector index after changing embedding model?",
     {("Vector Index Tuning", "parameters"): 3, ("Vector Index Tuning", "discussion"): 1}),

    ("Can similarity search and relational predicates run in one query?",
     {("Oracle AI Vector Search", "methods"): 3, ("Oracle AI Vector Search", "overview"): 2,
      ("Oracle AI Vector Search", "discussion"): 2}),

    ("How does the database generate embeddings without sending text elsewhere?",
     {("Oracle AI Vector Search", "methods"): 3, ("Oracle AI Vector Search", "discussion"): 1}),

    ("How does checkpointing differ from an agent's memory?",
     {("LangGraph", "discussion"): 3, ("LangGraph", "overview"): 2, ("LangGraph", "methods"): 1}),

    ("How is graph state persisted and resumed between runs?",
     {("LangGraph", "methods"): 3, ("LangGraph", "overview"): 2}),

    ("What happens to session state that was never promoted to durable storage?",
     {("In-Memory Session State", "methods"): 3, ("In-Memory Session State", "discussion"): 3,
      ("In-Memory Session State", "overview"): 1}),
]

In [21]:

cur.execute("""
SELECT d.title, JSON_VALUE(c.metadata,'$.section_type')
  FROM knowledge_base_chunk c JOIN knowledge_base_document d ON d.id = c.document_id
 WHERE JSON_VALUE(c.metadata,'$.section_type') <> 'references'
""")
ALL_CHUNKS = cur.fetchall()


def grade(title, section, labels):
    for (tp, st), g in labels.items():
        if title.startswith(tp) and section == st:
            return g
    return 0


def dcg(xs, k):
    return sum(g / math.log2(i + 2) for i, g in enumerate(xs[:k]))


def ndcg(gains, labels, k=10):
    """Ideal DCG must come from grading EVERY chunk, not from the label values.

    Labels are keyed by (title, section) and several chunks share a pair, so deriving the
    ideal from label values alone lets DCG exceed IDCG and produces NDCG > 1. A metric
    outside [0, 1] means the harness is broken, not the retriever.
    """
    ideal = sorted((grade(t, s, labels) for t, s in ALL_CHUNKS), reverse=True)
    d = dcg(ideal, k)
    return dcg(gains, k) / d if d else 0.0


def recall_at(gains, labels, k=20):
    total = sum(1 for t, s in ALL_CHUNKS if grade(t, s, labels) > 0)
    return (sum(1 for g in gains[:k] if g > 0) / total) if total else 0.0


def mrr(gains):
    return next((1.0 / i for i, g in enumerate(gains, 1) if g > 0), 0.0)


print(f"golden set: {len(GOLDEN_SET)} queries, "
      f"{sum(len(l) for _, l in GOLDEN_SET)} labels over {len(ALL_CHUNKS)} chunks")

golden set: 36 queries, 106 labels over 136 chunks


In [22]:

import math, statistics

EVAL_VECTOR = """
SELECT d.title, JSON_VALUE(c.metadata,'$.section_type')
  FROM knowledge_base_chunk c JOIN knowledge_base_document d ON d.id = c.document_id
 WHERE d.collection IN ('ingested-papers','vendor-docs')
   AND d.valid_until IS NULL AND d.deleted_at IS NULL AND c.deleted_at IS NULL
   AND (d.user_id IS NULL OR d.user_id = :user_id)
   AND JSON_VALUE(c.metadata,'$.section_type') <> 'references'
 ORDER BY VECTOR_DISTANCE(c.embedding,
          VECTOR_EMBEDDING(ALL_MINILM_L12_V2 USING :query AS DATA), COSINE)
 FETCH FIRST 20 ROWS ONLY
"""

EVAL_HYBRID = HYBRID_SQL.replace("FETCH FIRST 40 ROWS ONLY", "FETCH FIRST 20 ROWS ONLY")

EVAL_RERANK = RERANK_SQL.replace(
    "SELECT c.id, c.content, c.metadata, d.id AS document_id, d.title, d.source_uri, d.version,\n"
    "       k.rrf_score,",
    "SELECT d.title AS t, JSON_VALUE(c.metadata,'$.section_type') AS st,")

def evaluate(label, runner):
    nd, rc, rr, lat = [], [], [], []
    for q, labels in GOLDEN_SET:
        t0 = time.perf_counter()
        got = runner(q)
        lat.append((time.perf_counter() - t0) * 1000)
        gains = [grade(t, s, labels) for t, s in got]
        nd.append(ndcg(gains, labels, 10))
        rc.append(recall_at(gains, labels, 20))
        rr.append(mrr(gains))
    return (label, statistics.mean(nd), statistics.mean(rc), statistics.mean(rr),
            statistics.median(lat), statistics.stdev(nd), nd)


def run_vector(q):
    cur.execute(EVAL_VECTOR, query=q, user_id=USER_JANE)
    return cur.fetchall()


def run_hybrid(q):
    p = understand_query(q)
    cur.execute(EVAL_HYBRID, query=q, lex_query=build_lexical_query(p), user_id=USER_JANE)
    cs = [d[0].lower() for d in cur.description]
    return [(r[cs.index("title")], json.loads(r[cs.index("metadata")])["section_type"]
             if isinstance(r[cs.index("metadata")], str)
             else (r[cs.index("metadata")] or {}).get("section_type"))
            for r in cur.fetchall()]


def run_rerank(q):
    p = understand_query(q)
    cur.execute(RERANK_SQL, query=q, lex_query=build_lexical_query(p), user_id=USER_JANE)
    cs = [d[0].lower() for d in cur.description]
    out = []
    for r in cur.fetchall()[:20]:
        m = r[cs.index("metadata")]
        m = json.loads(m) if isinstance(m, str) else (m or {})
        out.append((r[cs.index("title")], m.get("section_type")))
    return out


# warm the reranker before timing anything
cur.execute("SELECT PREDICTION(BGE_RERANKER USING 'a' AS FIRST_INPUT,'b' AS SECOND_INPUT) FROM dual")
cur.fetchone()

results = [evaluate("Pure vector top-k", run_vector),
           evaluate("Hybrid (RRF)", run_hybrid),
           evaluate("Hybrid + rerank", run_rerank)]

print(f"{'Strategy':<20} {'NDCG@10':>9} {'Recall@20':>10} {'MRR':>7} {'p50 ms':>9}")
print("-" * 58)
for label, n, r, m, l, sd, _ in results:
    print(f"{label:<20} {n:>9.3f} {r:>10.3f} {m:>7.3f} {l:>9.0f}")

sd0 = results[0][5]
print(f"\nn = {len(GOLDEN_SET)}. Per-query sd of NDCG@10 is ~{sd0:.2f}, so the standard error")
print(f"is ~{sd0/len(GOLDEN_SET)**0.5:.3f}. Treat differences below that as noise.")

Strategy               NDCG@10  Recall@20     MRR    p50 ms
----------------------------------------------------------
Pure vector top-k        0.611      0.758   0.908         9
Hybrid (RRF)             0.559      0.689   0.865        19
Hybrid + rerank          0.714      0.783   0.958      2218

n = 36. Per-query sd of NDCG@10 is ~0.22, so the standard error
is ~0.036. Treat differences below that as noise.



### 9a. Read the table honestly

The `p50 ms` column reports total warm-path runtime for each strategy. The article's
`p50 added latency` column subtracts the 9 ms pure-vector baseline, which makes the saved
run +10 ms for Hybrid (RRF) and +2201 ms for Hybrid + rerank. A median describes the
evaluation distribution; it is not a per-query latency guarantee. Report p95 or an
observed range only after the evaluation cell records those values.

Two results here do **not** match the tidy story, and they are the most useful output of
this notebook.

**Reranking delivers the gain.** NDCG@10 and MRR both move substantially, and they move
without recall improving much, exactly as theory predicts, since a reranker only reorders
the candidate set it was handed.

**Hybrid retrieval does not help on this corpus.** It scores *below* pure vector. That is
not what the literature says, and it is worth understanding rather than hiding:

- The corpus is **137 chunks**. A 50-candidate vector pool already covers a third of it,
  so there is very little for a second retriever to rescue.
- The chunks average **25 words**. Oracle Text `SCORE()` is a coarse integer and, on text
  this short, produces heavy ties. One query here matched 80 chunks across only 10 distinct
  scores. Ranking within a tie group is arbitrary, and RRF reads that arbitrary order as
  signal. `DENSE_RANK` (used above) mitigates this; it does not eliminate it.
- RRF weights both retrievers **equally**. When one signal is strong and the other is
  weak and noisy, equal weighting dilutes the strong one.

Hybrid retrieval earns its reputation on large corpora of substantial passages, where the
vector pool genuinely misses exact-match content. This corpus is neither. The honest conclusion is not "hybrid is useless". It is **"measure it on your corpus, because the result is not guaranteed."** That is the whole argument for this section.

In [23]:

# Segment by whether the query names a rare identifier -- the case hybrid exists to fix.
IDENT = re.compile(r"\b(Letta|AgentCore|MemGPT|LangGraph|Oracle|BM25|NDCG|MRR)\b")
seg = {"names an identifier": [], "purely conceptual": []}
for i, (q, _) in enumerate(GOLDEN_SET):
    seg["names an identifier" if IDENT.search(q) else "purely conceptual"].append(i)

print(f"{'segment':<22} {'n':>3}" + "".join(f"{lbl[:18]:>20}" for lbl, *_ in results))
print("-" * 85)
for label, idxs in seg.items():
    line = f"{label:<22} {len(idxs):>3}"
    for r in results:
        per_q = r[6]
        line += f"{statistics.mean([per_q[i] for i in idxs]):>20.3f}"
    print(line)

print("\nHybrid is closest to breaking even exactly where it should be: on queries naming")
print("a rare identifier that embeddings under-weight. It still does not win here, but the")
print("gap narrows, which is the mechanism showing through a corpus too small to exploit it.")

segment                  n   Pure vector top-k        Hybrid (RRF)     Hybrid + rerank
-------------------------------------------------------------------------------------
names an identifier      7               0.537               0.506               0.780
purely conceptual       29               0.629               0.572               0.698

Hybrid is closest to breaking even exactly where it should be: on queries naming
a rare identifier that embeddings under-weight. It still does not win here, but the
gap narrows, which is the mechanism showing through a corpus too small to exploit it.



## 10. Wiring the pipeline into LangChain and LangGraph

LangChain gives the retrieval pipeline a standard interface; it does not define the
pipeline itself.

All database access goes through a manager class, because part 2's rule has not changed:
the manager is the only code allowed to touch the schema, and a framework adapter does not
get an exemption.

In [24]:

class MemoryManager:
    """The one door into the schema. Part 2's rule, minimally realised."""

    def __init__(self, connection):
        self.conn = connection

    def understand_query(self, question: str) -> dict:
        p = understand_query(question)
        p["lexical_query"] = build_lexical_query(p)
        return p

    def search_knowledge_base(self, plan: dict, *, user_id: str, top_k: int = 40):
        """Stages 2-5 in one statement: filter, both pools, RRF, cross-encoder rescore."""
        c = self.conn.cursor()
        c.execute(RERANK_SQL, query=plan["question"],
                  lex_query=plan["lexical_query"], user_id=user_id)
        cols_ = [d[0].lower() for d in c.description]
        return [dict(zip(cols_, r)) for r in c.fetchall()][:top_k]


from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever


class OracleHybridRetriever(BaseRetriever):
    """LangChain entry point. This class only adapts an interface; it owns no SQL."""

    manager: object
    user_id: str
    k: int = 8
    candidates: int = 40
    max_tokens: int = 1200

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list:
        plan = self.manager.understand_query(query)                      # stage 1
        rows = self.manager.search_knowledge_base(                       # stages 2-5
            plan, user_id=self.user_id, top_k=self.candidates)
        ws, _ = budget_context(rows, max_tokens=self.max_tokens,         # stage 6
                               diversify_by=plan["diversify_by"])
        return [Document(page_content=r["content"],
                         metadata={"title": r["title"], "source": r["source_uri"],
                                   "document_id": r["document_id"],
                                   "version": r["version"],
                                   "score": r["rerank_score"]})
                for r in ws[: self.k]]


manager = MemoryManager(conn)
retriever = OracleHybridRetriever(manager=manager, user_id=USER_JANE, k=6)
docs = retriever.invoke(plan["question"])

print(f"retriever returned {len(docs)} documents\n")
for d in docs:
    print(f"  [{d.metadata['score']:>7.3f}] {d.metadata['title'][:44]:<44} "
          f"v{d.metadata['version']}")
print("\nProvenance rides in metadata, which is what lets the agent cite a source")
print("rather than gesture at the corpus.")

retriever returned 6 documents

  [ -0.036] Letta: Agent Memory Beyond the Context Windo v1
  [ -0.762] Letta: Agent Memory Beyond the Context Windo v1
  [ -2.933] Letta: Agent Memory Beyond the Context Windo v1
  [ -3.716] Amazon Bedrock AgentCore Memory Developer Gu v1
  [ -3.779] Amazon Bedrock AgentCore Memory Developer Gu v1
  [ -3.864] Amazon Bedrock AgentCore Memory Developer Gu v1

Provenance rides in metadata, which is what lets the agent cite a source
rather than gesture at the corpus.



When the pipeline needs observable stages, and in production it does, LangGraph is the better frame. One node per stage, explicit state between nodes, a graph you can checkpoint
and instrument.

Note the graph has **four nodes for six stages**. Stages 2 through 5 are one node because
they are one SQL statement; re-separating them in the orchestration layer would add round
trips to recreate a boundary the database deliberately erased.

In [25]:

from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class RetrievalState(TypedDict):
    query: str
    plan: dict
    candidates: list
    working_set: list


def n_understand(state):
    return {"plan": manager.understand_query(state["query"])}


def n_retrieve(state):
    return {"candidates": manager.search_knowledge_base(
        state["plan"], user_id=USER_JANE, top_k=40)}


def n_budget(state):
    ws, used = budget_context(state["candidates"],
                              max_tokens=state["plan"]["token_budget"],
                              diversify_by=state["plan"]["diversify_by"])
    return {"working_set": ws}


g = StateGraph(RetrievalState)
g.add_node("understand", n_understand)   # stage 1
g.add_node("retrieve", n_retrieve)       # stages 2-5: one SQL statement
g.add_node("budget", n_budget)           # stage 6
g.add_edge(START, "understand")
g.add_edge("understand", "retrieve")
g.add_edge("retrieve", "budget")
g.add_edge("budget", END)
retrieval_graph = g.compile()

out = retrieval_graph.invoke({"query": plan["question"]})
print("graph nodes:", list(retrieval_graph.get_graph().nodes)[1:-1])
print(f"intent: {out['plan']['intent']}   candidates: {len(out['candidates'])}   "
      f"working set: {len(out['working_set'])}")
print("\nworking set:")
for r in out["working_set"]:
    print(f"  [{r['rerank_score']:>7.3f}] {r['title'][:46]}")

graph nodes: ['understand', 'retrieve', 'budget']
intent: comparison   candidates: 40   working set: 7

working set:
  [ -0.036] Letta: Agent Memory Beyond the Context Window
  [ -0.762] Letta: Agent Memory Beyond the Context Window
  [ -2.933] Letta: Agent Memory Beyond the Context Window
  [ -3.716] Amazon Bedrock AgentCore Memory Developer Guid
  [ -3.779] Amazon Bedrock AgentCore Memory Developer Guid
  [ -3.864] Amazon Bedrock AgentCore Memory Developer Guid
  [ -5.292] Forgetting Mechanisms in Neural Memory Systems



## 11. Cleanup

Drops everything this notebook created. The reranking model is left in place. It is expensive to produce and shared across runs.

In [26]:

for stmt in [
    "BEGIN DBMS_RLS.DROP_POLICY(USER, 'KNOWLEDGE_BASE_CHUNK', 'KB_CHUNK_TENANT_POL'); END;",
    "BEGIN DBMS_RLS.DROP_POLICY(USER, 'KNOWLEDGE_BASE_DOCUMENT', 'KB_DOC_TENANT_POL'); END;",
    "DROP TABLE knowledge_base_chunk CASCADE CONSTRAINTS PURGE",
    "DROP TABLE knowledge_base_document CASCADE CONSTRAINTS PURGE",
    "DROP CONTEXT memory_ctx",
    "DROP PACKAGE set_memory_ctx",
    "DROP FUNCTION memory_tenant_policy",
]:
    try:
        cur.execute(stmt)
    except oracledb.DatabaseError as e:
        print(f"  skip: {str(e)[:60]}")

conn.commit()
print("\nDropped tables, policies, context and package.")
print("BGE_RERANKER left loaded (DBMS_VECTOR.DROP_ONNX_MODEL to remove it).")
conn.close()


Dropped tables, policies, context and package.
BGE_RERANKER left loaded (DBMS_VECTOR.DROP_ONNX_MODEL to remove it).
